# Part 7 — Applied ML for Deployment

*CinemaStream — The Forward Deployed Engineer's Handbook*

---

In [ ]:
# ── CinemaStream: one-time setup ──────────────────────────────────────────────
# Run this cell FIRST if you are on Google Colab or a fresh local environment.
# Skip it if you have already cloned the repo and installed requirements.
#
# !pip install -r requirements.txt
# !git clone https://github.com/YOUR_ORG/cinemastream.git
# import os; os.chdir("cinemastream")
# ─────────────────────────────────────────────────────────────────────────────
# Ensure the canonical dataset exists (deterministic; safe to re-run).
try:
    from cinemastream.scripts.generate_data import generate
    generate()
except ModuleNotFoundError:
    print("Run the clone/cd lines above first (Colab), then re-run this cell.")

## Chapters in this notebook

- [Chapter 67: Supervised Learning — Regression vs Classification, KNN, SVM](#chapter_67_supervised_learning_regression_vs_classification_knn_svm)
- [Chapter 67a: Regression Methods — From Least Squares to Splines](#chapter_67a_regression_methods_from_least_squares_to_splines)
- [Chapter 68: Decision Trees and Ensemble Methods — Bagging, Feature Importance](#chapter_68_decision_trees_and_ensemble_methods_bagging_feature_importance)
- [Chapter 69: Feature Engineering — Encoding, Scaling, Binning, and Feature Selection](#chapter_69_feature_engineering_encoding_scaling_binning_and_feature_selection)
- [Chapter 70: Handling Missing Data and Outliers — Imputation, Capping, and Robust Scalers Revisited](#chapter_70_handling_missing_data_and_outliers_imputation_capping_and_robust_scalers_revisited)
- [Chapter 71: Train/Validation/Test Split — Stratification, Time-Series Splits, and Cross-Validation](#chapter_71_train_validation_test_split_stratification_time_series_splits_and_cross_validation)
- [Chapter 72: Metrics — Accuracy, Precision, Recall, F1, AUC-ROC, Log-Loss, MAE, and MSE](#chapter_72_metrics_accuracy_precision_recall_f1_auc_roc_log_loss_mae_and_mse)
- [Chapter 72a: Causal Thinking for Data & ML Engineers](#chapter_72a_causal_thinking_for_data_ml_engineers)
- [Chapter 73: Hyperparameter Tuning — Grid Search, Random Search, and Bayesian Optimization with Optuna](#chapter_73_hyperparameter_tuning_grid_search_random_search_and_bayesian_optimization_with_optuna)
- [Chapter 74: Model Comparison and the Bias-Variance Trade-off — Under/Overfitting and Learning Curves](#chapter_74_model_comparison_and_the_bias_variance_trade_off_under_overfitting_and_learning_curves)
- [Chapter 74a: Unsupervised Learning — Clustering, PCA & Anomaly Detection](#chapter_74a_unsupervised_learning_clustering_pca_anomaly_detection)
- [Chapter 74b: Recommender Systems — Collaborative, Content-Based & Hybrid](#chapter_74b_recommender_systems_collaborative_content_based_hybrid)
- [Chapter 75: ML Experiment Tracking — MLflow, the Model Registry, Artifact Management, and Reproducibility](#chapter_75_ml_experiment_tracking_mlflow_the_model_registry_artifact_management_and_reproducibility)
- [Chapter 75a: Data & Model Versioning — DVC, Artifacts & Reproducibility](#chapter_75a_data_model_versioning_dvc_artifacts_reproducibility)
- [Chapter 76: Model Serialization — pickle, joblib, and ONNX](#chapter_76_model_serialization_pickle_joblib_and_onnx)
- [Chapter 77: ML Pipeline Integration — REST APIs with FastAPI, and Putting the Churn Model in Production](#chapter_77_ml_pipeline_integration_rest_apis_with_fastapi_and_putting_the_churn_model_in_production)
- [Chapter 77a: Feature Stores — Offline/Online Features & Training-Serving Skew](#chapter_77a_feature_stores_offline_online_features_training_serving_skew)
- [Chapter 77b: Time-Series Forecasting for Business Metrics](#chapter_77b_time_series_forecasting_for_business_metrics)

---

# Chapter 67: Supervised Learning — Regression vs Classification, KNN, SVM

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    A[Labeled training data] --> B{What to predict?}
    B -->|Continuous number| C[Regression\ne.g., watch minutes]
    B -->|Discrete category| D[Classification\ne.g., churn yes/no]
    D --> E{Algorithm choice}
    E --> F[KNN\nVote from K nearest neighbors]
    E --> G[SVM\nFind max-margin boundary]
    F --> H[Sensitive to feature scale\nand outlier neighbors]
    G --> I[Kernel trick for\nnon-linear boundaries]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas scikit-learn

### 2.1 Regression: predicting a continuous number

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Toy regression: predicting house price (in thousands) from size (sq meters)
sizes = np.array([50, 65, 70, 85, 100, 110, 120, 140]).reshape(-1, 1)
prices = np.array([150, 190, 200, 240, 280, 300, 330, 380])

reg = LinearRegression()
reg.fit(sizes, prices)

print("Slope (price per sq meter):", round(reg.coef_[0], 2))
print("Intercept:", round(reg.intercept_, 2))

new_size = np.array([[90]])
predicted_price = reg.predict(new_size)
print("Predicted price for 90 sq m:", round(predicted_price[0], 2))

```
Slope (price per sq meter): 2.55
Intercept: 22.88
Predicted price for 90 sq m: 252.38
```

### 2.2 Classification: predicting a category

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

X = np.array([
    [150, 0],
    [170, 0],
    [140, 0],
    [130, 1],
    [180, 1],
    [200, 1],
    [120, 0],
    [190, 1],
])
y = np.array([0, 0, 0, 1, 1, 1, 0, 1])  # 0=apple, 1=orange

clf = LogisticRegression()
clf.fit(X, y)

new_fruit = np.array([[160, 1]])
prediction = clf.predict(new_fruit)
probability = clf.predict_proba(new_fruit)

print("Prediction (0=apple, 1=orange):", prediction[0])
print("Probability [apple, orange]:", np.round(probability[0], 3))

```
Prediction (0=apple, 1=orange): 1
Probability [apple, orange]: [0.356 0.644]
```

### 2.3 KNN: classification by neighbors

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

accuracy = knn.score(X_test, y_test)
print("KNN (k=5) test accuracy:", round(accuracy, 3))

for k in [1, 3, 5, 9, 15]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    print(f"  k={k}: accuracy={acc:.3f}")

```
KNN (k=5) test accuracy: 0.9
  k=1: accuracy=0.820
  k=3: accuracy=0.940
  k=5: accuracy=0.900
  k=9: accuracy=0.880
  k=15: accuracy=0.860
```

### 2.4 SVM and the scaling requirement

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# SVMs are distance-based -- scale features first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for kernel in ["linear", "rbf"]:
    svm = SVC(kernel=kernel, random_state=42)
    svm.fit(X_train_scaled, y_train)
    acc = svm.score(X_test_scaled, y_test)
    n_support = svm.n_support_
    print(f"SVM (kernel={kernel}): accuracy={acc:.3f}, support vectors per class={n_support}")

```
SVM (kernel=linear): accuracy=0.880, support vectors per class=[33 33]
SVM (kernel=rbf): accuracy=0.920, support vectors per class=[36 34]
```

## 3. CinemaStream in Practice

In [ ]:
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
N = 300

countries = rng.choice(
    ["VN", "PH", "ID", "MY", "TH", "SG", "IN"],
    size=N,
    p=[0.20, 0.18, 0.18, 0.12, 0.12, 0.10, 0.10],
)
plans = rng.choice(["Free", "Basic", "Premium"], size=N, p=[0.35, 0.40, 0.25])

# Canonical churn rates (overall / Premium) from the team's "Ask Anything" work,
# Chapters 064-066: VN 8.2%/3.1%, PH 6.5%/5.9%, ID 5.0%/2.0%.
# Other countries use an invented baseline of 5.5% overall / 2.5% Premium.
OVERALL_CHURN = {"VN": 0.082, "PH": 0.065, "ID": 0.050, "MY": 0.055, "TH": 0.055, "SG": 0.055, "IN": 0.055}
PREMIUM_CHURN = {"VN": 0.031, "PH": 0.059, "ID": 0.020, "MY": 0.025, "TH": 0.025, "SG": 0.025, "IN": 0.025}

def base_churn_prob(country, plan):
    if plan == "Premium":
        return PREMIUM_CHURN[country]
    return OVERALL_CHURN[country]

tenure_months = rng.integers(1, 37, size=N)
watch_minutes_avg = np.clip(rng.normal(loc=85, scale=30, size=N), 5, None)
days_since_last_watch = rng.integers(0, 60, size=N)
support_tickets_count = rng.poisson(0.6, size=N)

rows = []
for i in range(N):
    base_p = base_churn_prob(countries[i], plans[i])
    # Behavioral multiplier: keeps each country/plan group centered near its
    # canonical base rate while giving the model real per-row signal.
    multiplier = 1.0
    if days_since_last_watch[i] > 21:
        multiplier *= 1.8
    if watch_minutes_avg[i] < 40:
        multiplier *= 1.6
    if support_tickets_count[i] >= 2:
        multiplier *= 1.5
    if tenure_months[i] < 3:
        multiplier *= 1.3
    if tenure_months[i] > 24:
        multiplier *= 0.6
    adj_p = np.clip(base_p * multiplier, 0.01, 0.95)
    churned = rng.random() < adj_p
    rows.append({
        "user_id": 1000 + i,
        "country": countries[i],
        "plan": plans[i],
        "watch_minutes_avg": round(float(watch_minutes_avg[i]), 1),
        "days_since_last_watch": int(days_since_last_watch[i]),
        "tenure_months": int(tenure_months[i]),
        "support_tickets_count": int(support_tickets_count[i]),
        "churned": int(churned),
    })

churn_df = pd.DataFrame(rows)
print(churn_df.head())
print("\nShape:", churn_df.shape)
print("\nOverall churn rate:", round(churn_df["churned"].mean(), 3))
print("\nChurn rate by country:")
print(churn_df.groupby("country")["churned"].mean().round(3))

```
   user_id country     plan  ...  tenure_months  support_tickets_count  churned
0     1000      TH  Premium  ...             32                      0        0
1     1001      ID     Free  ...             18                      1        0
2     1002      SG    Basic  ...             36                      0        0
3     1003      TH    Basic  ...             28                      0        0
4     1004      VN  Premium  ...             24                      2        0

[5 rows x 8 columns]

Shape: (300, 8)

Overall churn rate: 0.07

Churn rate by country:
country
ID    0.058
IN    0.154
MY    0.029
PH    0.085
SG    0.032
TH    0.108
VN    0.056
Name: churned, dtype: float64
```

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Encode categorical features (country, plan) as one-hot columns
features_df = pd.get_dummies(
    churn_df.drop(columns=["user_id", "churned"]),
    columns=["country", "plan"],
    drop_first=True,
)
X = features_df.values
y = churn_df["churned"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)
print("Train churn rate:", round(y_train.mean(), 3), "| Test churn rate:", round(y_test.mean(), 3))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
log_reg.fit(X_train_scaled, y_train)
log_reg_acc = log_reg.score(X_test_scaled, y_test)
print("\nLogistic Regression test accuracy:", round(log_reg_acc, 3))

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_acc = knn.score(X_test_scaled, y_test)
print("KNN (k=5) test accuracy:", round(knn_acc, 3))

majority_class = int(round(y_train.mean()))
baseline_preds = np.full_like(y_test, majority_class)
baseline_acc = accuracy_score(y_test, baseline_preds)
print("Majority-class baseline accuracy:", round(baseline_acc, 3))

```
Train churn rate: 0.071 | Test churn rate: 0.067

Logistic Regression test accuracy: 0.933
KNN (k=5) test accuracy: 0.933
Majority-class baseline accuracy: 0.933
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

sizes = np.array([50, 65, 70, 85, 100, 110, 120, 140]).reshape(-1, 1)
prices = np.array([150, 190, 200, 240, 280, 300, 330, 380])
reg = LinearRegression().fit(sizes, prices)

print(reg.predict(np.array([[200]])))

```
[532.875]
```

In [ ]:
premium_churn_by_country = churn_df[churn_df["plan"] == "Premium"].groupby("country")["churned"].agg(["mean", "count"])
premium_churn_by_country["mean"] = premium_churn_by_country["mean"].round(3)
print(premium_churn_by_country)

```
          mean  count
country              
ID       0.071     14
IN       0.000      3
MY       0.083     12
PH       0.167     18
SG       0.111      9
TH       0.000     11
VN       0.000      8
```

---

# Chapter 67a: Regression Methods — From Least Squares to Splines

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Ordinary Least Squares (OLS) — the baseline

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

rng = np.random.default_rng(42)
X_raw = rng.uniform(20, 150, size=40)
y = 2.5 * X_raw + 10 + rng.normal(0, 15, size=40)

X = X_raw.reshape(-1, 1)

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

print(f"Coefficient (slope):  {model.coef_[0]:.3f}")
print(f"Intercept:            {model.intercept_:.3f}")
print(f"R²:                   {r2_score(y, y_pred):.3f}")
print(f"RMSE:                 {mean_squared_error(y, y_pred) ** 0.5:.3f}")

```
Coefficient (slope):  2.432
Intercept:            16.256
R²:                   0.987
RMSE:                 10.510
```

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
X_raw = rng.uniform(20, 150, size=40)
y = 2.5 * X_raw + 10 + rng.normal(0, 15, size=40)
X = X_raw.reshape(-1, 1)

model = LinearRegression().fit(X, y)
residuals = y - model.predict(X)

print("Residuals summary:")
print(f"  Mean:   {residuals.mean():.4f}  (should be ~0)")
print(f"  Std:    {residuals.std():.3f}")
print(f"  Min:    {residuals.min():.3f}")
print(f"  Max:    {residuals.max():.3f}")

```
Residuals summary:
  Mean:   -0.0000  (should be ~0)
  Std:    10.510
  Min:    -23.462
  Max:    18.086
```

### 2.2 Polynomial Regression — bending the line

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

rng = np.random.default_rng(42)
X_raw = np.linspace(-3, 3, 50)
y = 0.5 * X_raw**3 - X_raw**2 + rng.normal(0, 0.5, 50)
X = X_raw.reshape(-1, 1)

for degree in [1, 3, 10]:
    pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    pipe.fit(X, y)
    r2 = r2_score(y, pipe.predict(X))
    print(f"degree={degree:2d}  train R²={r2:.4f}")

```
degree= 1  train R²=0.6636
degree= 3  train R²=0.9964
degree=10  train R²=0.9968
```

### 2.3 Weighted Least Squares (WLS) — not all errors are equal

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

rng = np.random.default_rng(42)
n = 60
X_raw = rng.uniform(1, 100, size=n)
y = 3.0 * X_raw + 5 + rng.normal(0, X_raw * 0.3, size=n)
X = X_raw.reshape(-1, 1)

weights = 1.0 / (X_raw ** 2)
weights = weights / weights.sum() * n

ols = LinearRegression().fit(X, y)
wls = LinearRegression().fit(X, y, sample_weight=weights)

print(f"OLS  — coef: {ols.coef_[0]:.3f}, intercept: {ols.intercept_:.3f}")
print(f"WLS  — coef: {wls.coef_[0]:.3f}, intercept: {wls.intercept_:.3f}")

```
OLS  — coef: 2.938, intercept: 4.863
WLS  — coef: 2.959, intercept: 4.480
```

### 2.4 Ridge Regression (L2 regularisation)

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

rng = np.random.default_rng(42)
n, p = 80, 10
X = rng.standard_normal((n, p))
true_coef = rng.uniform(-3, 3, size=p)
y = X @ true_coef + rng.normal(0, 2, size=n)

ols = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
ridge_low = make_pipeline(StandardScaler(), Ridge(alpha=0.1)).fit(X, y)
ridge_high = make_pipeline(StandardScaler(), Ridge(alpha=100)).fit(X, y)

for name, m in [("OLS", ols), ("Ridge α=0.1", ridge_low), ("Ridge α=100", ridge_high)]:
    coefs = m.named_steps[list(m.named_steps)[-1]].coef_
    r2 = r2_score(y, m.predict(X))
    print(f"{name:15s}  train R²={r2:.3f}  |coef| max={np.abs(coefs).max():.3f}  min={np.abs(coefs).min():.3f}")

```
OLS              train R²=0.763  |coef| max=2.708  min=0.111
Ridge α=0.1      train R²=0.763  |coef| max=2.703  min=0.109
Ridge α=100      train R²=0.501  |coef| max=1.054  min=0.056
```

### 2.5 Lasso Regression (L1 regularisation) — built-in feature selection

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(42)
n, p = 100, 12
X = rng.standard_normal((n, p))
# Only the first 4 features actually matter
true_coef = np.array([2.0, -1.5, 3.0, -0.8] + [0.0] * 8)
y = X @ true_coef + rng.normal(0, 1, size=n)

lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.2, max_iter=5000)).fit(X, y)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X, y)

lasso_coef = lasso.named_steps["lasso"].coef_
ridge_coef = ridge.named_steps["ridge"].coef_

print("Feature  True    Lasso     Ridge")
for i, (t, l, r) in enumerate(zip(true_coef, lasso_coef, ridge_coef)):
    marker = " <-- noise" if t == 0 else ""
    print(f"  f{i:02d}    {t:6.1f}  {l:8.4f}  {r:8.4f}{marker}")

print(f"\nLasso zeros: {(lasso_coef == 0).sum()} of {p}")
print(f"Ridge zeros: {(ridge_coef == 0).sum()} of {p}")

```
Feature  True    Lasso     Ridge
  f00       2.0    1.8519    2.1059
  f01      -1.5   -1.4519   -1.6302
  f02       3.0    2.8325    3.1506
  f03      -0.8   -0.4856   -0.7306
  f04       0.0   -0.0000    0.0026 <-- noise
  f05       0.0    0.0174    0.2371 <-- noise
  f06       0.0    0.0000   -0.0366 <-- noise
  f07       0.0    0.0000    0.0970 <-- noise
  f08       0.0   -0.0000   -0.0539 <-- noise
  f09       0.0    0.0000    0.0856 <-- noise
  f10       0.0   -0.0000   -0.1579 <-- noise
  f11       0.0    0.0105    0.1615 <-- noise

Lasso zeros: 6 of 12
Ridge zeros: 0 of 12
```

### 2.6 Non-Linear Least Squares — fitting a known equation

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

def saturation_curve(x, Vmax, Km):
    """Michaelis-Menten / saturation model: y = Vmax * x / (Km + x)"""
    return Vmax * x / (Km + x)

rng = np.random.default_rng(42)
x_data = np.linspace(0.5, 20, 30)
y_true = saturation_curve(x_data, Vmax=50.0, Km=5.0)
y_noisy = y_true + rng.normal(0, 2.0, size=30)

popt, pcov = curve_fit(saturation_curve, x_data, y_noisy, p0=[40, 3])
perr = np.sqrt(np.diag(pcov))

print(f"Vmax: fitted={popt[0]:.3f}  true=50.000  ±{perr[0]:.3f}")
print(f"Km:   fitted={popt[1]:.3f}  true= 5.000  ±{perr[1]:.3f}")

```
Vmax: fitted=51.555  true=50.000  ±1.413
Km:   fitted=5.451  true= 5.000  ±0.439
```

### 2.7 Spline Fitting and Smoothing Splines

In [ ]:
import numpy as np
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score
from scipy.interpolate import UnivariateSpline

rng = np.random.default_rng(42)
x = np.linspace(0, 4 * np.pi, 80)
y = np.sin(x) + 0.5 * np.cos(2 * x) + rng.normal(0, 0.15, 80)
X = x.reshape(-1, 1)

# Approach 1: sklearn SplineTransformer (B-spline basis) + Ridge
spline_model = make_pipeline(
    SplineTransformer(n_knots=8, degree=3),
    Ridge(alpha=0.01)
)
spline_model.fit(X, y)
r2_spline = r2_score(y, spline_model.predict(X))

# Approach 2: scipy UnivariateSpline (smoothing spline, s controls smoothness)
us = UnivariateSpline(x, y, s=2.0)
r2_smooth = r2_score(y, us(x))

print(f"SplineTransformer + Ridge  R²={r2_spline:.4f}")
print(f"UnivariateSpline (s=2.0)   R²={r2_smooth:.4f}")

```
SplineTransformer + Ridge  R²=0.8355
UnivariateSpline (s=2.0)   R²=0.9601
```

### 2.8 Least Absolute Deviations (LAD) — robustness to outliers

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, HuberRegressor
import statsmodels.formula.api as smf
import pandas as pd

rng = np.random.default_rng(42)
x = rng.uniform(0, 10, 50)
y = 2.0 * x + 5.0 + rng.normal(0, 1.5, 50)

# Inject 5 severe outliers
x_all = np.append(x, [2.0, 3.0, 4.0, 5.0, 6.0])
y_all = np.append(y, [60.0, 65.0, 70.0, 68.0, 72.0])

X_all = x_all.reshape(-1, 1)

ols = LinearRegression().fit(X_all, y_all)
huber = HuberRegressor(epsilon=1.35).fit(X_all, y_all)

df = pd.DataFrame({"y": y_all, "x": x_all})
lad = smf.quantreg("y ~ x", df).fit(q=0.5, disp=False)

print(f"True:  slope=2.0,  intercept=5.0")
print(f"OLS:   slope={ols.coef_[0]:.3f}, intercept={ols.intercept_:.3f}")
print(f"Huber: slope={huber.coef_[0]:.3f}, intercept={huber.intercept_:.3f}")
print(f"LAD:   slope={lad.params['x']:.3f}, intercept={lad.params['Intercept']:.3f}")

```
True:  slope=2.0,  intercept=5.0
OLS:   slope=1.210, intercept=13.814
Huber: slope=1.978, intercept=5.114
LAD:   slope=1.893, intercept=5.571
```

### 2.9 Orthogonal Distance Regression (ODR) — error in X too

In [ ]:
import numpy as np
from scipy.odr import ODR, Model, RealData

rng = np.random.default_rng(42)
x_true = np.linspace(1, 10, 40)
y_true = 2.5 * x_true + 1.0

x_obs = x_true + rng.normal(0, 0.8, 40)
y_obs = y_true + rng.normal(0, 1.5, 40)

def linear_fn(params, x):
    return params[0] * x + params[1]

odr_model = Model(linear_fn)
data = RealData(x_obs, y_obs, sx=0.8, sy=1.5)
odr_fit = ODR(data, odr_model, beta0=[2.0, 0.5]).run()

from sklearn.linear_model import LinearRegression
ols = LinearRegression().fit(x_obs.reshape(-1, 1), y_obs)

print(f"True:  slope=2.5, intercept=1.0")
print(f"OLS:   slope={ols.coef_[0]:.3f}, intercept={ols.intercept_:.3f}")
print(f"ODR:   slope={odr_fit.beta[0]:.3f}, intercept={odr_fit.beta[1]:.3f}")

```
True:  slope=2.5, intercept=1.0
OLS:   slope=2.235, intercept=2.409
ODR:   slope=2.372, intercept=1.651
```

### 2.10 Deming Regression — known error ratio

In [ ]:
import numpy as np
from scipy.odr import ODR, Model, RealData

rng = np.random.default_rng(42)
x_true = np.linspace(5, 50, 35)
y_true = 1.2 * x_true + 3.0
x_obs = x_true + rng.normal(0, 2.0, 35)
y_obs = y_true + rng.normal(0, 2.0, 35)

# Deming regression: delta = sigma_x^2 / sigma_y^2
# Here both errors have std=2.0, so delta=1 (equal error ratio)
delta = 1.0

def linear_fn(params, x):
    return params[0] * x + params[1]

data = RealData(x_obs, y_obs, sx=np.sqrt(delta), sy=1.0)
odr_fit = ODR(data, Model(linear_fn), beta0=[1.0, 0.0]).run()

from sklearn.linear_model import LinearRegression
ols = LinearRegression().fit(x_obs.reshape(-1, 1), y_obs)

print(f"True:    slope=1.200, intercept=3.000")
print(f"OLS:     slope={ols.coef_[0]:.3f}, intercept={ols.intercept_:.3f}")
print(f"Deming:  slope={odr_fit.beta[0]:.3f}, intercept={odr_fit.beta[1]:.3f}")

```
True:    slope=1.200, intercept=3.000
OLS:     slope=1.158, intercept=4.203
Deming:  slope=1.170, intercept=3.870
```

### 2.11 Random Forest Regression — ensemble fitting

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

rng = np.random.default_rng(42)
n = 200
x1 = rng.uniform(0, 10, n)
x2 = rng.uniform(0, 10, n)
# Non-linear truth: trees should beat Ridge here
y = np.sin(x1) * np.log1p(x2) + 0.5 * x1 - 0.3 * x2 + rng.normal(0, 0.3, n)

X = np.column_stack([x1, x2])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
ridge = Ridge(alpha=1.0)

rf.fit(X_train, y_train)
ridge.fit(X_train, y_train)

for name, m in [("RandomForest", rf), ("Ridge", ridge)]:
    r2 = r2_score(y_test, m.predict(X_test))
    rmse = mean_squared_error(y_test, m.predict(X_test)) ** 0.5
    print(f"{name:15s}  test R²={r2:.3f}  RMSE={rmse:.3f}")

print(f"\nFeature importances (RF): x1={rf.feature_importances_[0]:.3f}, x2={rf.feature_importances_[1]:.3f}")

```
RandomForest     test R²=0.943  RMSE=0.503
Ridge            test R²=0.619  RMSE=1.301

Feature importances (RF): x1=0.838, x2=0.162
```

## 3. CinemaStream in Practice

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score

# Regenerate the same churn_df from Ch 067 (identical seed and logic)
rng = np.random.default_rng(42)
N = 300
countries = rng.choice(
    ["VN", "PH", "ID", "MY", "TH", "SG", "IN"], size=N,
    p=[0.20, 0.18, 0.18, 0.12, 0.12, 0.10, 0.10]
)
plans = rng.choice(["Free", "Basic", "Premium"], size=N, p=[0.35, 0.40, 0.25])
tenure_months = rng.integers(1, 37, size=N)
watch_minutes_avg = np.clip(rng.normal(loc=85, scale=30, size=N), 5, None)
days_since_last_watch = rng.integers(0, 60, size=N)
support_tickets_count = rng.poisson(0.6, size=N)

churn_df = pd.DataFrame({
    "country": countries,
    "plan": plans,
    "watch_minutes_avg": watch_minutes_avg.round(1),
    "days_since_last_watch": days_since_last_watch,
    "tenure_months": tenure_months,
    "support_tickets_count": support_tickets_count,
})

# Target: watch_minutes_avg
# Features: everything else
y = churn_df["watch_minutes_avg"].values
feature_df = churn_df.drop(columns=["watch_minutes_avg"])

num_features = ["days_since_last_watch", "tenure_months", "support_tickets_count"]
cat_features = ["plan", "country"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_features),
])

models = {
    "OLS":       make_pipeline(preprocessor, LinearRegression()),
    "Ridge":     make_pipeline(preprocessor, Ridge(alpha=1.0)),
    "Lasso":     make_pipeline(preprocessor, Lasso(alpha=0.5, max_iter=5000)),
    "RF":        make_pipeline(preprocessor, RandomForestRegressor(n_estimators=100, random_state=42)),
}

print(f"{'Model':10s}  {'CV R² mean':>12s}  {'CV R² std':>10s}")
print("-" * 38)
for name, pipe in models.items():
    scores = cross_val_score(pipe, feature_df, y, cv=5, scoring="r2")
    print(f"{name:10s}  {scores.mean():12.3f}  {scores.std():10.3f}")

```
Model         CV R² mean   CV R² std
--------------------------------------
OLS               -0.050       0.057
Ridge             -0.046       0.055
Lasso             -0.028       0.031
RF                -0.149       0.060
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

# Inject real signal: watch minutes depends on tenure + plan
rng2 = np.random.default_rng(99)
n = 300
tenure = rng2.integers(1, 37, size=n)
plan_enc = rng2.choice([0, 1, 2], size=n)  # 0=Free, 1=Basic, 2=Premium
noise_feat = rng2.standard_normal(size=n)

watch_signal = 50 + 1.2 * tenure + 15 * (plan_enc == 2) + rng2.normal(0, 12, n)

signal_df = pd.DataFrame({
    "tenure_months": tenure,
    "plan_code": plan_enc,
    "noise_feature": noise_feat,
})

prep2 = StandardScaler()
X_scaled = prep2.fit_transform(signal_df)

lasso2 = Lasso(alpha=2.0, max_iter=5000).fit(X_scaled, watch_signal)

print("Lasso coefficients (signal-injected data):")
for feat, coef in zip(signal_df.columns, lasso2.coef_):
    status = "ZEROED" if coef == 0.0 else "kept  "
    print(f"  {feat:20s}  {coef:8.3f}  [{status}]")

```
Lasso coefficients (signal-injected data):
  tenure_months            9.841  [kept  ]
  plan_code                3.985  [kept  ]
  noise_feature           -0.000  [ZEROED]
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(42)
n, p = 80, 20
X = rng.standard_normal((n, p))
# Only 3 features are real; rest are noise
true_coef = np.array([3.0, -2.0, 1.5] + [0.0] * 17)
y = X @ true_coef + rng.normal(0, 1.5, n)

pipe = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=10000, random_state=42))
pipe.fit(X, y)

lasso = pipe.named_steps["lassocv"]
print(f"Best alpha:       {lasso.alpha_:.4f}")
print(f"Non-zero coefs:  {(lasso.coef_ != 0).sum()} of {p}")
print(f"Zero coefs:      {(lasso.coef_ == 0).sum()} of {p}")
print(f"CV R² (best):    {lasso.score(pipe.named_steps['standardscaler'].transform(X), y):.3f}")

```
Best alpha:       0.1312
Non-zero coefs:  10 of 20
Zero coefs:      10 of 20
CV R² (best):    0.890
```

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(7)
x = np.linspace(0, 10, 50)
y = 4 * np.sin(x) + x + rng.normal(0, 0.5, 50)
X = x.reshape(-1, 1)

poly3 = make_pipeline(PolynomialFeatures(3), LinearRegression())
spline6 = make_pipeline(SplineTransformer(n_knots=6, degree=3), Ridge(alpha=0.1))

for name, m in [("Poly deg-3 + OLS", poly3), ("Spline 6-knot + Ridge", spline6)]:
    cv_r2 = cross_val_score(m, X, y, cv=5, scoring="r2")
    print(f"{name:25s}  CV R²={cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

```
Poly deg-3 + OLS           CV R²=-92.164 ± 118.126
Spline 6-knot + Ridge      CV R²=0.044 ± 1.054
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(42)
N = 300
watch_minutes_avg = np.clip(rng.normal(loc=85, scale=30, size=N), 5, None)
days_since_last_watch = rng.integers(0, 60, size=N)
tenure_months = rng.integers(1, 37, size=N)
support_tickets_count = rng.poisson(0.6, size=N)

X_raw = pd.DataFrame({
    "watch_minutes_avg": watch_minutes_avg.round(1),
    "days_since_last_watch": days_since_last_watch,
    "support_tickets_count": support_tickets_count,
})
y_tenure = tenure_months

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

for name, m in [("OLS", LinearRegression()), ("Ridge α=10", Ridge(alpha=10)), ("Lasso α=1", Lasso(alpha=1, max_iter=5000))]:
    cv_r2 = cross_val_score(m, X_scaled, y_tenure, cv=5, scoring="r2")
    print(f"{name:12s}  CV R²={cv_r2.mean():.3f}")

lasso = Lasso(alpha=1, max_iter=5000).fit(X_scaled, y_tenure)
print("\nLasso coefficients:")
for feat, coef in zip(X_raw.columns, lasso.coef_):
    print(f"  {feat:30s}  {coef:.4f}  {'ZERO' if coef == 0 else ''}")

```
OLS           CV R²=-0.031
Ridge α=10    CV R²=-0.029
Lasso α=1     CV R²=-0.008

Lasso coefficients:
  watch_minutes_avg               0.0000  ZERO
  days_since_last_watch           0.0000  ZERO
  support_tickets_count           0.0000  ZERO
```

---

# Chapter 68: Decision Trees and Ensemble Methods — Bagging, Feature Importance

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    A[Training data] --> B[Single Decision Tree]
    B --> C{max_depth = None?}
    C -->|Yes| D[Memorizes training set\nOverfits badly]
    C -->|No| E[Generalizes better\nLower training accuracy]
    A --> F[Bagging: bootstrap samples]
    F --> G[Tree 1\ndifferent sample]
    F --> H[Tree 2\ndifferent sample]
    F --> I[Tree N\ndifferent sample]
    G --> J[Majority vote\nRandom Forest]
    H --> J
    I --> J
    J --> K[Feature importance\nranked by impurity reduction]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install numpy scikit-learn

### 2.1 A single tree, and how it overfits

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

for max_depth in [None, 2, 3, 5]:
    tree = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    tree.fit(X_train, y_train)
    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)
    print(f"max_depth={str(max_depth):>4}: train_acc={train_acc:.3f}, test_acc={test_acc:.3f}, gap={train_acc - test_acc:.3f}")

```
max_depth=None: train_acc=1.000, test_acc=0.820, gap=0.180
max_depth=   2: train_acc=0.847, test_acc=0.880, gap=-0.033
max_depth=   3: train_acc=0.860, test_acc=0.880, gap=-0.020
max_depth=   5: train_acc=0.933, test_acc=0.820, gap=0.113
```

### 2.2 Bagging and Random Forest

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Single unconstrained tree (the overfit case from 2.1)
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
print(f"Single tree (unconstrained): train_acc={tree.score(X_train, y_train):.3f}, test_acc={tree.score(X_test, y_test):.3f}")

# Random Forest of 100 such trees
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f"Random Forest (100 trees): train_acc={rf.score(X_train, y_train):.3f}, test_acc={rf.score(X_test, y_test):.3f}")

print()
print("Feature importances:", np.round(rf.feature_importances_, 3))

```
Single tree (unconstrained): train_acc=1.000, test_acc=0.820
Random Forest (100 trees): train_acc=1.000, test_acc=0.960
Feature importances: [0.704 0.296]
```

## 3. CinemaStream in Practice

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from cinemastream.ml.churn.train import build_churn_dataset, prepare_features, RANDOM_SEED

churn_df = build_churn_dataset()
X, y, feature_names = prepare_features(churn_df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)
print("Train churn rate:", round(y_train.mean(), 3), "| Test churn rate:", round(y_test.mean(), 3))
print("Feature names:", feature_names)

```
Train churn rate: 0.071 | Test churn rate: 0.067
Feature names: ['watch_minutes_avg', 'days_since_last_watch', 'tenure_months', 'support_tickets_count', 'country_IN', 'country_MY', 'country_PH', 'country_SG', 'country_TH', 'country_VN', 'plan_Free', 'plan_Premium']
```

In [ ]:
# Decision Tree -- default (unconstrained), then a shallower version
tree_default = DecisionTreeClassifier(random_state=RANDOM_SEED)
tree_default.fit(X_train, y_train)
print(f"Decision Tree (default, unconstrained): train_acc={tree_default.score(X_train, y_train):.3f}, test_acc={tree_default.score(X_test, y_test):.3f}")

tree_shallow = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
tree_shallow.fit(X_train, y_train)
print(f"Decision Tree (max_depth=3):            train_acc={tree_shallow.score(X_train, y_train):.3f}, test_acc={tree_shallow.score(X_test, y_test):.3f}")

print()

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
rf.fit(X_train, y_train)
print(f"Random Forest (100 trees):              train_acc={rf.score(X_train, y_train):.3f}, test_acc={rf.score(X_test, y_test):.3f}")

majority_class = int(round(y_train.mean()))
baseline_preds = np.full_like(y_test, majority_class)
baseline_acc = accuracy_score(y_test, baseline_preds)
print(f"Majority-class baseline:                {baseline_acc:.3f}")

```
Decision Tree (default, unconstrained): train_acc=1.000, test_acc=0.907
Decision Tree (max_depth=3):            train_acc=0.942, test_acc=0.920

Random Forest (100 trees):              train_acc=1.000, test_acc=0.933
Majority-class baseline:                0.933
```

In [ ]:
importances = rf.feature_importances_
order = np.argsort(importances)[::-1]
print("Random Forest feature importances (descending):")
for i in order:
    print(f"  {feature_names[i]:25s} {importances[i]:.3f}")

```
Random Forest feature importances (descending):
  watch_minutes_avg         0.295
  days_since_last_watch     0.233
  tenure_months             0.223
  support_tickets_count     0.063
  plan_Premium              0.036
  plan_Free                 0.034
  country_PH                0.033
  country_IN                0.024
  country_SG                0.019
  country_TH                0.016
  country_VN                0.012
  country_MY                0.010
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import numpy as np

country_features = [(name, imp) for name, imp in zip(feature_names, rf.feature_importances_) if name.startswith("country_")]
country_features.sort(key=lambda pair: pair[1])
print(country_features[0])

```
('country_MY', np.float64(0.010424185242464438))
```

---

# Chapter 69: Feature Engineering — Encoding, Scaling, Binning, and Feature Selection

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas scikit-learn

### 2.1 Encoding categorical variables

In [ ]:
import pandas as pd

products = pd.DataFrame({
    "product": ["Widget A", "Widget B", "Widget C", "Widget D"],
    "size": ["small", "large", "medium", "small"],
    "color": ["red", "blue", "green", "blue"],
})
print(products)

```
    product    size  color
0  Widget A   small    red
1  Widget B   large   blue
2  Widget C  medium  green
3  Widget D   small   blue
```

In [ ]:
size_order = {"small": 0, "medium": 1, "large": 2}
products["size_encoded"] = products["size"].map(size_order)
print(products[["product", "size", "size_encoded"]])

```
    product    size  size_encoded
0  Widget A   small             0
1  Widget B   large             2
2  Widget C  medium             1
3  Widget D   small             0
```

In [ ]:
products_encoded = pd.get_dummies(products[["product", "color"]], columns=["color"], prefix="color")
print(products_encoded)

```
    product  color_blue  color_green  color_red
0  Widget A       False        False       True
1  Widget B        True        False      False
2  Widget C       False         True      False
3  Widget D        True        False      False
```

### 2.2 Scaling numeric features

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

prices = np.array([5.5, 7.2, 9.1, 6.8, 1000.0]).reshape(-1, 1)

for name, scaler in [("StandardScaler", StandardScaler()), ("MinMaxScaler", MinMaxScaler()), ("RobustScaler", RobustScaler())]:
    scaled = scaler.fit_transform(prices)
    print(f"{name:15s}", np.round(scaled.flatten(), 3))

```
StandardScaler  [-0.504 -0.5   -0.495 -0.501  2.   ]
MinMaxScaler    [0.    0.002 0.004 0.001 1.   ]
RobustScaler    [-7.39000e-01  0.00000e+00  8.26000e-01 -1.74000e-01  4.31652e+02]
```

### 2.3 Binning continuous variables

In [ ]:
ages = pd.Series([5, 12, 17, 22, 35, 47, 62, 78], name="age")
equal_width = pd.cut(ages, bins=3, labels=["young", "middle", "old"])
equal_freq = pd.qcut(ages, q=3, labels=["young", "middle", "old"])
binned = pd.DataFrame({"age": ages, "equal_width": equal_width, "equal_freq": equal_freq})
print(binned)
print()
print("equal_width bin edges:", pd.cut(ages, bins=3, retbins=True)[1].round(2))
print("equal_freq bin edges:", pd.qcut(ages, q=3, retbins=True)[1].round(2))

```
   age equal_width equal_freq
0    5       young      young
1   12       young      young
2   17       young      young
3   22       young     middle
4   35      middle     middle
5   47      middle        old
6   62         old        old
7   78         old        old

equal_width bin edges: [ 4.93 29.33 53.67 78.  ]
equal_freq bin edges: [ 5.   18.67 43.   78.  ]
```

### 2.4 Feature selection

In [ ]:
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(n_samples=200, n_features=6, n_informative=3, random_state=42)

selector = SelectKBest(score_func=f_classif, k=3)
selector.fit(X, y)
print("SelectKBest scores:        ", np.round(selector.scores_, 2))
print("SelectKBest selected idx:  ", selector.get_support(indices=True))

rfe = RFE(RandomForestClassifier(random_state=42, n_estimators=50), n_features_to_select=3)
rfe.fit(X, y)
print("RFE ranking (1=selected):  ", rfe.ranking_)
print("RFE selected idx:          ", np.where(rfe.support_)[0])

```
SelectKBest scores:         [ 55.03  29.68   0.   161.9    0.36  90.7 ]
SelectKBest selected idx:   [0 3 5]
RFE ranking (1=selected):   [1 2 4 1 3 1]
RFE selected idx:           [0 3 5]
```

## 3. CinemaStream in Practice

### 3.1 Engineering two new features

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd
from cinemastream.ml.churn.train import build_churn_dataset, prepare_features, RANDOM_SEED


def engineer_features(churn_df):
    df = churn_df.copy()

    # Binning: tenure_months -> tenure_bucket, mirroring the behavioral
    # risk multipliers used to generate this dataset (Chapter 067):
    # tenure < 3 -> "new" (1.3x risk), tenure > 24 -> "veteran" (0.6x risk)
    df["tenure_bucket"] = pd.cut(
        df["tenure_months"],
        bins=[0, 3, 12, 24, 37],
        labels=["new", "growing", "established", "veteran"],
        right=False,
    )

    # Derived ratio feature: combines watch_minutes_avg and
    # days_since_last_watch into one "engagement" number.
    # +1 avoids division by zero for users who watched today.
    df["engagement_score"] = (
        df["watch_minutes_avg"] / (df["days_since_last_watch"] + 1)
    ).round(3)

    return df


churn_df = build_churn_dataset()
engineered_df = engineer_features(churn_df)
print(engineered_df[
    ["user_id", "tenure_months", "tenure_bucket", "watch_minutes_avg",
     "days_since_last_watch", "engagement_score"]
].head())
print()
print("tenure_bucket value counts:")
print(engineered_df["tenure_bucket"].value_counts().sort_index())

```
   user_id  tenure_months  ... days_since_last_watch  engagement_score
0     1000             32  ...                    16             4.829
1     1001             18  ...                    40             2.898
2     1002             36  ...                    54             0.302
3     1003             28  ...                    54             0.729
4     1004             24  ...                    44             1.273

[5 rows x 6 columns]

tenure_bucket value counts:
tenure_bucket
new             20
growing         71
established     99
veteran        110
Name: count, dtype: int64
```

### 3.2 Scaling: does CinemaStream's data need RobustScaler?

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

raw = churn_df["watch_minutes_avg"].values.reshape(-1, 1)
print("Raw watch_minutes_avg:  min={:.1f}  max={:.1f}  mean={:.1f}".format(
    raw.min(), raw.max(), raw.mean()
))

for name, scaler in [("StandardScaler", StandardScaler()), ("RobustScaler", RobustScaler())]:
    scaled = scaler.fit_transform(raw)
    print(f"{name:15s} min={scaled.min():.3f}  max={scaled.max():.3f}  mean={scaled.mean():.3f}")

```
Raw watch_minutes_avg:  min=5.0  max=180.4  mean=85.1
StandardScaler  min=-2.707  max=3.224  mean=-0.000
RobustScaler    min=-2.154  max=2.499  mean=-0.030
```

### 3.3 Feature selection: do the new features earn their place?

In [ ]:
def prepare_features_v2(churn_df):
    engineered_df = engineer_features(churn_df)
    features_df = pd.get_dummies(
        engineered_df.drop(columns=["user_id", "churned"]),
        columns=["country", "plan", "tenure_bucket"],
        drop_first=True,
    )
    X = features_df.values
    y = engineered_df["churned"].values
    return X, y, list(features_df.columns)


X_v1, y, feature_names_v1 = prepare_features(churn_df)
X_v2, y, feature_names_v2 = prepare_features_v2(churn_df)
print(f"v1 features: {len(feature_names_v1)} | v2 features: {len(feature_names_v2)}")

```
v1 features: 12 | v2 features: 16
```

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier

selector = SelectKBest(score_func=f_classif, k=8)
selector.fit(X_v2, y)
order = np.argsort(selector.scores_)[::-1]
print("SelectKBest (f_classif) -- top 8 of 16 features:")
for i in order[:8]:
    print(f"  {feature_names_v2[i]:25s} score={selector.scores_[i]:.2f}")

print()
rfe = RFE(RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED), n_features_to_select=8)
rfe.fit(X_v2, y)
rfe_selected = [feature_names_v2[i] for i in np.where(rfe.support_)[0]]
print("RFE (RandomForest) -- top 8 of 16 features:")
for name in rfe_selected:
    print(f"  {name}")

```
SelectKBest (f_classif) -- top 8 of 16 features:
  tenure_months             score=7.30
  tenure_bucket_veteran     score=7.29
  country_IN                score=3.09
  tenure_bucket_growing     score=1.16
  country_MY                score=1.04
  country_TH                score=0.94
  country_SG                score=0.75
  watch_minutes_avg         score=0.70

RFE (RandomForest) -- top 8 of 16 features:
  watch_minutes_avg
  days_since_last_watch
  tenure_months
  support_tickets_count
  engagement_score
  country_TH
  country_VN
  tenure_bucket_established
```

### 3.4 Does any of this beat 0.933?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def run_rf_model(X, y, feature_names, label):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
    )
    rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
    rf.fit(X_train, y_train)
    train_acc = rf.score(X_train, y_train)
    test_acc = rf.score(X_test, y_test)
    majority_class = int(round(y_train.mean()))
    baseline_preds = np.full_like(y_test, majority_class)
    baseline_acc = accuracy_score(y_test, baseline_preds)
    print(f"{label:32s} ({len(feature_names):2d} features): "
          f"train_acc={train_acc:.3f}, test_acc={test_acc:.3f}")
    return rf, test_acc, baseline_acc

_, v1_acc, baseline_acc = run_rf_model(X_v1, y, feature_names_v1, "v1 (Chapter 068 baseline)")
_, v2_acc, _ = run_rf_model(X_v2, y, feature_names_v2, "v2 (all engineered features)")

sub_idx = [feature_names_v2.index(n) for n in rfe_selected]
X_v2_sub = X_v2[:, sub_idx]
_, v2_sub_acc, _ = run_rf_model(X_v2_sub, y, rfe_selected, "v2 (RFE-selected subset)")

print(f"{'Majority-class baseline':32s} ({'--':>2s} features): test_acc={baseline_acc:.3f}")

```
v1 (Chapter 068 baseline)        (12 features): train_acc=1.000, test_acc=0.933
v2 (all engineered features)     (16 features): train_acc=1.000, test_acc=0.920
v2 (RFE-selected subset)         ( 8 features): train_acc=1.000, test_acc=0.933
Majority-class baseline          (-- features): test_acc=0.933
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
rfe_selected = ['watch_minutes_avg', 'days_since_last_watch', 'tenure_months',
                'support_tickets_count', 'engagement_score', 'country_TH',
                'country_VN', 'tenure_bucket_established']

ch068_top3 = {"watch_minutes_avg", "days_since_last_watch", "tenure_months"}
print("Chapter 068's top 3 all present in RFE selection:", ch068_top3.issubset(set(rfe_selected)))

```
Chapter 068's top 3 all present in RFE selection: True
```

---

# Chapter 70: Handling Missing Data and Outliers — Imputation, Capping, and Robust Scalers Revisited

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas scikit-learn

### 2.1 Detecting missing data

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "product": ["Widget A", "Widget B", "Widget C", "Widget D", "Widget E"],
    "price": [12.5, np.nan, 9.0, 45.0, np.nan],
    "rating": [4.2, 3.8, np.nan, 4.9, 4.1],
})
print(df)
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Missing fraction per column:")
print(df.isnull().mean().round(2))

```
    product  price  rating
0  Widget A   12.5     4.2
1  Widget B    NaN     3.8
2  Widget C    9.0     NaN
3  Widget D   45.0     4.9
4  Widget E    NaN     4.1

Missing values per column:
product    0
price      2
rating     1
dtype: int64

Missing fraction per column:
product    0.0
price      0.4
rating     0.2
dtype: float64
```

### 2.2 Imputation: filling in the gaps

In [ ]:
from sklearn.impute import SimpleImputer

imputer_mean = SimpleImputer(strategy="mean")
df["price_mean_imputed"] = imputer_mean.fit_transform(df[["price"]])

imputer_median = SimpleImputer(strategy="median")
df["rating_median_imputed"] = imputer_median.fit_transform(df[["rating"]])

print(df[["product", "price", "price_mean_imputed", "rating", "rating_median_imputed"]])

```
    product  price  price_mean_imputed  rating  rating_median_imputed
0  Widget A   12.5           12.500000     4.2                   4.20
1  Widget B    NaN           22.166667     3.8                   3.80
2  Widget C    9.0            9.000000     NaN                   4.15
3  Widget D   45.0           45.000000     4.9                   4.90
4  Widget E    NaN           22.166667     4.1                   4.10
```

In [ ]:
df2 = pd.DataFrame({
    "category": ["Electronics", "Electronics", "Electronics", "Clothing", "Clothing", "Clothing"],
    "price": [299.0, np.nan, 350.0, 15.0, np.nan, 22.0],
})
df2["price_global_median"] = df2["price"].fillna(df2["price"].median())
df2["price_group_median"] = df2.groupby("category")["price"].transform(lambda s: s.fillna(s.median()))
print(df2)

```
      category  price  price_global_median  price_group_median
0  Electronics  299.0                299.0                299.0
1  Electronics    NaN                160.5                324.5
2  Electronics  350.0                350.0                350.0
3     Clothing   15.0                 15.0                 15.0
4     Clothing    NaN                160.5                 18.5
5     Clothing   22.0                 22.0                 22.0
```

### 2.3 Outlier detection: the IQR method

In [ ]:
values = pd.Series([12, 14, 15, 13, 16, 14, 15, 95, 13, 14], name="value")
Q1 = values.quantile(0.25)
Q3 = values.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Bounds: [{lower_bound}, {upper_bound}]")
outliers = values[(values < lower_bound) | (values > upper_bound)]
print("Outliers:", outliers.tolist())

```
Q1=13.25, Q3=15.0, IQR=1.75
Bounds: [10.625, 17.625]
Outliers: [95]
```

### 2.4 Capping (winsorizing)

In [ ]:
capped = values.clip(lower=lower_bound, upper=upper_bound)
print("Original:", values.tolist())
print("Capped:  ", capped.tolist())
print()
print(f"Mean before capping: {values.mean():.2f}, std before: {values.std():.2f}")
print(f"Mean after capping:  {capped.mean():.2f}, std after:  {capped.std():.2f}")

```
Original: [12, 14, 15, 13, 16, 14, 15, 95, 13, 14]
Capped:   [12.0, 14.0, 15.0, 13.0, 16.0, 14.0, 15.0, 17.625, 13.0, 14.0]

Mean before capping: 22.10, std before: 25.64
Mean after capping:  14.36, std after:  1.63
```

## 3. CinemaStream in Practice

### 3.1 Injecting a realistic billing column

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from cinemastream.ml.churn.train import build_churn_dataset, RANDOM_SEED

PLAN_BASE_SPEND = {"Free": 0.0, "Basic": 8.90, "Premium": 12.90}


def inject_billing_data(churn_df, random_seed=RANDOM_SEED):
    rng = np.random.default_rng(random_seed + 1)  # separate stream from build_churn_dataset
    df = churn_df.copy()
    n = len(df)

    base = df["plan"].map(PLAN_BASE_SPEND).to_numpy()
    noise = rng.normal(0, 0.5, size=n)
    spend = np.round(base + noise, 2)
    spend = np.clip(spend, 0, None)

    # Double-charge bug: ~5% of Premium users billed twice + a small fee
    premium_idx = np.where(df["plan"].to_numpy() == "Premium")[0]
    n_bugged = max(1, int(len(premium_idx) * 0.05))
    bugged_idx = rng.choice(premium_idx, size=n_bugged, replace=False)
    spend[bugged_idx] = np.round(spend[bugged_idx] * 2 + rng.uniform(0, 5, size=n_bugged), 2)

    # ~8% missing, independent of plan (MCAR)
    missing_idx = rng.choice(n, size=int(n * 0.08), replace=False)
    spend[missing_idx] = np.nan

    df["monthly_spend_sgd"] = spend
    return df


churn_df = build_churn_dataset()
billed_df = inject_billing_data(churn_df)
print(billed_df[["user_id", "plan", "monthly_spend_sgd"]].head(8))

```
   user_id     plan  monthly_spend_sgd
0     1000  Premium              13.02
1     1001     Free               0.34
2     1002    Basic                NaN
3     1003    Basic               8.45
4     1004  Premium              11.90
5     1005    Basic               9.39
6     1006    Basic               8.91
7     1007    Basic               9.00
```

### 3.2 Checking the missingness pattern

In [ ]:
def detect_missingness(df):
    missing_count = df["monthly_spend_sgd"].isnull().sum()
    missing_pct = df["monthly_spend_sgd"].isnull().mean()
    print(f"monthly_spend_sgd missing: {missing_count} of {len(df)} ({missing_pct:.1%})")
    print()
    print("Missing rate by plan:")
    print(df.groupby("plan")["monthly_spend_sgd"].apply(lambda s: s.isnull().mean()).round(3))


detect_missingness(billed_df)

```
monthly_spend_sgd missing: 24 of 300 (8.0%)

Missing rate by plan:
plan
Basic      0.113
Free       0.069
Premium    0.040
```

### 3.3 Imputation: global median vs group-aware median

In [ ]:
def impute_spend(df):
    df = df.copy()
    global_median = df["monthly_spend_sgd"].median()
    df["spend_global_imputed"] = df["monthly_spend_sgd"].fillna(global_median)
    df["spend_group_imputed"] = df.groupby("plan")["monthly_spend_sgd"].transform(
        lambda s: s.fillna(s.median())
    )
    return df, global_median


imputed_df, global_median = impute_spend(billed_df)
print(f"Global median spend (all plans): {global_median}")
print()
print("Per-plan MEAN of spend_global_imputed vs spend_group_imputed:")
print(imputed_df.groupby("plan")[["spend_global_imputed", "spend_group_imputed"]].mean().round(2))

```
Global median spend (all plans): 8.83

Per-plan MEAN of spend_global_imputed vs spend_group_imputed:
         spend_global_imputed  spend_group_imputed
plan                                              
Basic                    8.90                 8.90
Free                     0.80                 0.19
Premium                 13.38                13.54
```

### 3.4 Outlier detection and capping

In [ ]:
def detect_and_cap_outliers(df, column="spend_group_imputed"):
    df = df.copy()
    Q1, Q3 = df[column].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    n_outliers = ((df[column] < lower) | (df[column] > upper)).sum()
    df[f"{column}_capped"] = df[column].clip(lower, upper)
    return df, lower, upper, n_outliers


capped_df, lower, upper, n_outliers = detect_and_cap_outliers(imputed_df)
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")
print(f"Outliers detected: {n_outliers}")
print()
premium_outliers = capped_df[
    (capped_df["plan"] == "Premium") & (capped_df["spend_group_imputed"] > upper)
][["user_id", "plan", "spend_group_imputed", "spend_group_imputed_capped"]]
print(premium_outliers)
print()
print(f"spend_group_imputed        -- mean={capped_df['spend_group_imputed'].mean():.2f}, "
      f"std={capped_df['spend_group_imputed'].std():.2f}, max={capped_df['spend_group_imputed'].max():.2f}")
print(f"spend_group_imputed_capped -- mean={capped_df['spend_group_imputed_capped'].mean():.2f}, "
      f"std={capped_df['spend_group_imputed_capped'].std():.2f}, max={capped_df['spend_group_imputed_capped'].max():.2f}")

```
IQR bounds: [-15.28, 26.32]
Outliers detected: 3

     user_id     plan  spend_group_imputed  spend_group_imputed_capped
60      1060  Premium                30.57                     26.3175
167     1167  Premium                27.23                     26.3175
271     1271  Premium                28.64                     26.3175

spend_group_imputed        -- mean=7.13, std=5.52, max=30.57
spend_group_imputed_capped -- mean=7.11, std=5.43, max=26.32
```

### 3.5 Does the new feature change the model?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from cinemastream.ml.churn.features import engineer_features

CH069_RFE_FEATURES = [
    "watch_minutes_avg", "days_since_last_watch", "tenure_months",
    "support_tickets_count", "engagement_score",
    "country_TH", "country_VN", "tenure_bucket_established",
]


def prepare_features_v3(df):
    engineered_df = engineer_features(df)
    features_df = pd.get_dummies(
        engineered_df.drop(columns=["user_id", "churned"]),
        columns=["country", "plan", "tenure_bucket"],
        drop_first=True,
    )
    for col in CH069_RFE_FEATURES:
        if col not in features_df.columns:
            features_df[col] = 0

    X_v2 = features_df[CH069_RFE_FEATURES].to_numpy()
    X_v3 = features_df[CH069_RFE_FEATURES + ["spend_group_imputed_capped"]].to_numpy()
    y = engineered_df["churned"].to_numpy()
    return X_v2, X_v3, y, CH069_RFE_FEATURES, CH069_RFE_FEATURES + ["spend_group_imputed_capped"]


def run_rf_model(X, y, feature_names, label):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
    )
    rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
    rf.fit(X_train, y_train)
    print(f"{label:38s} ({len(feature_names)} features): "
          f"train_acc={rf.score(X_train, y_train):.3f}, test_acc={rf.score(X_test, y_test):.3f}")


X_v2, X_v3, y, names_v2, names_v3 = prepare_features_v3(capped_df)
run_rf_model(X_v2, y, names_v2, "v2 (Ch069 RFE-selected, 8 features)")
run_rf_model(X_v3, y, names_v3, "v3 (+ spend_group_imputed_capped)")

```
v2 (Ch069 RFE-selected, 8 features)    (8 features): train_acc=1.000, test_acc=0.933
v3 (+ spend_group_imputed_capped)      (9 features): train_acc=1.000, test_acc=0.933
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import pandas as pd

durations = pd.Series([8, 9, 7, 10, 9, 8, 11, 9, 0.5, 60])
Q1, Q3 = durations.quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Bounds: [{lower}, {upper}]")
outliers = durations[(durations < lower) | (durations > upper)]
print("Outliers:", outliers.tolist())
print("Capped:", durations.clip(lower, upper).tolist())

```
Q1=8.0, Q3=9.75, IQR=1.75
Bounds: [5.375, 12.375]
Outliers: [0.5, 60.0]
Capped: [8.0, 9.0, 7.0, 10.0, 9.0, 8.0, 11.0, 9.0, 5.375, 12.375]
```

---

# Chapter 71: Train/Validation/Test Split — Stratification, Time-Series Splits, and Cross-Validation

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    A[Full dataset\nstratify=y] --> B[Train 60%]
    A --> C[Validation 20%]
    A --> D[Test 20%\nnever touched until final eval]
    B --> E[Fit model]
    C --> F[Tune hyperparameters\ncompare max_depth options]
    F --> E
    E --> G{StratifiedKFold CV}
    G --> H[Fold 1 score]
    G --> I[Fold 2 score]
    G --> J[Fold k score]
    H --> K[Mean ± std\nhow stable is 0.933?]
    I --> K
    J --> K
    D --> L[Final honest score\nreported once]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install numpy scikit-learn

### 2.1 A three-way train/validation/test split

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=300, n_features=4, n_informative=3, n_redundant=0,
                            random_state=42)

# First split off the test set (20% of everything)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
# Then split what's left into train (75% of it = 60% overall) and val (25% of it = 20% overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

print(f"Total rows: {len(X)}")
print(f"Train: {len(X_train)} ({len(X_train)/len(X):.0%})")
print(f"Val:   {len(X_val)} ({len(X_val)/len(X):.0%})")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X):.0%})")

```
Total rows: 300
Train: 180 (60%)
Val:   60 (20%)
Test:  60 (20%)
```

### 2.2 Stratification on an imbalanced dataset

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X_imb, y_imb = make_classification(
    n_samples=200, n_features=4, n_informative=3, n_redundant=0,
    weights=[0.9, 0.1], random_state=42
)
print(f"Overall class distribution: {np.bincount(y_imb)} "
      f"({np.bincount(y_imb)[1] / len(y_imb):.1%} class 1)")
print()

# Without stratify
Xtr_ns, Xte_ns, ytr_ns, yte_ns = train_test_split(X_imb, y_imb, test_size=0.25, random_state=0)
print("WITHOUT stratify=y:")
print(f"  Train class counts: {np.bincount(ytr_ns)} ({np.bincount(ytr_ns)[1]/len(ytr_ns):.1%} class 1)")
print(f"  Test  class counts: {np.bincount(yte_ns)} ({np.bincount(yte_ns)[1]/len(yte_ns):.1%} class 1)")
print()

# With stratify
Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(X_imb, y_imb, test_size=0.25, random_state=0, stratify=y_imb)
print("WITH stratify=y:")
print(f"  Train class counts: {np.bincount(ytr_s)} ({np.bincount(ytr_s)[1]/len(ytr_s):.1%} class 1)")
print(f"  Test  class counts: {np.bincount(yte_s)} ({np.bincount(yte_s)[1]/len(yte_s):.1%} class 1)")

```
Overall class distribution: [180  20] (10.0% class 1)

WITHOUT stratify=y:
  Train class counts: [133  17] (11.3% class 1)
  Test  class counts: [47  3] (6.0% class 1)

WITH stratify=y:
  Train class counts: [135  15] (10.0% class 1)
  Test  class counts: [45  5] (10.0% class 1)
```

### 2.3 KFold vs StratifiedKFold cross-validation

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Same imbalanced toy dataset as Section 2.2
X_imb, y_imb = make_classification(
    n_samples=200, n_features=4, n_informative=3, n_redundant=0,
    weights=[0.9, 0.1], random_state=42
)

clf = LogisticRegression(max_iter=1000)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
kfold_scores = cross_val_score(clf, X_imb, y_imb, cv=kfold)
print(f"KFold(5) scores:           {np.round(kfold_scores, 3)}")
print(f"KFold(5) mean +/- std:     {kfold_scores.mean():.3f} +/- {kfold_scores.std():.3f}")
print()

skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skfold_scores = cross_val_score(clf, X_imb, y_imb, cv=skfold)
print(f"StratifiedKFold(5) scores: {np.round(skfold_scores, 3)}")
print(f"StratifiedKFold(5) mean +/- std: {skfold_scores.mean():.3f} +/- {skfold_scores.std():.3f}")
print()

# Per-fold test-set class-1 proportion
print("Per-fold test-set class-1 proportion:")
print("  KFold:           ", end="")
for _, test_idx in KFold(n_splits=5, shuffle=True, random_state=42).split(X_imb, y_imb):
    print(f"{y_imb[test_idx].mean():.2f}", end="  ")
print()
print("  StratifiedKFold: ", end="")
for _, test_idx in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_imb, y_imb):
    print(f"{y_imb[test_idx].mean():.2f}", end="  ")
print()

```
KFold(5) scores:           [1.    1.    0.925 0.975 0.95 ]
KFold(5) mean +/- std:     0.970 +/- 0.029

StratifiedKFold(5) scores: [0.95  0.975 0.975 0.975 0.975]
StratifiedKFold(5) mean +/- std: 0.970 +/- 0.010

Per-fold test-set class-1 proportion:
  KFold:            0.05  0.15  0.12  0.10  0.07  
  StratifiedKFold:  0.10  0.10  0.10  0.10  0.10
```

### 2.4 TimeSeriesSplit vs a random split on time-ordered data

In [ ]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, KFold

rng = np.random.default_rng(42)
months = np.arange(24)
revenue = 1000 + 50 * months + rng.normal(0, 30, size=24)
print("Monthly revenue (first 6 / last 6):")
print("  ", np.round(revenue[:6], 1))
print("  ", np.round(revenue[-6:], 1))
print()

tscv = TimeSeriesSplit(n_splits=4)
print("TimeSeriesSplit(n_splits=4) fold boundaries:")
for i, (train_idx, test_idx) in enumerate(tscv.split(months)):
    print(f"  Fold {i}: train=months[{train_idx[0]}:{train_idx[-1]+1}] "
          f"({len(train_idx)} months), test=months[{test_idx[0]}:{test_idx[-1]+1}] "
          f"({len(test_idx)} months)")
print()

print("Contrast: a random KFold(4) shuffle would mix future months into training folds:")
kf = KFold(n_splits=4, shuffle=True, random_state=42)
for i, (train_idx, test_idx) in enumerate(kf.split(months)):
    print(f"  Fold {i}: test months = {sorted(test_idx.tolist())}")

```
Monthly revenue (first 6 / last 6):
   [1009.1 1018.8 1122.5 1178.2 1141.5 1210.9]
   [1926.4 1948.5 1994.5 2029.6 2136.7 2145.4]

TimeSeriesSplit(n_splits=4) fold boundaries:
  Fold 0: train=months[0:8] (8 months), test=months[8:12] (4 months)
  Fold 1: train=months[0:12] (12 months), test=months[12:16] (4 months)
  Fold 2: train=months[0:16] (16 months), test=months[16:20] (4 months)
  Fold 3: train=months[0:20] (20 months), test=months[20:24] (4 months)

Contrast: a random KFold(4) shuffle would mix future months into training folds:
  Fold 0: test months = [0, 8, 9, 11, 16, 18]
  Fold 1: test months = [1, 2, 5, 12, 13, 21]
  Fold 2: test months = [3, 4, 15, 17, 20, 22]
  Fold 3: test months = [6, 7, 10, 14, 19, 23]
```

## 3. CinemaStream in Practice

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

from cinemastream.ml.churn.train import build_churn_dataset, RANDOM_SEED
from cinemastream.ml.churn.data_quality import (
    inject_billing_data, impute_spend, detect_and_cap_outliers,
    prepare_features_v3, run_rf_model,
)
from cinemastream.ml.churn.cross_validation import run_cross_validation

churn_df = build_churn_dataset()
billed_df = inject_billing_data(churn_df)
imputed_df, _ = impute_spend(billed_df)
capped_df, _, _, _ = detect_and_cap_outliers(imputed_df)

X_v2, X_v3, y, names_v2, names_v3 = prepare_features_v3(capped_df)

print("--- Single 75/25 split (Ch067-070 baseline) ---")
run_rf_model(X_v3, y, names_v3, "v3 (9 features), single split")
print()

print("--- StratifiedKFold(5) cross-validation ---")
run_cross_validation(X_v2, y, names_v2, "v2 (8 features, Ch069 RFE)")
run_cross_validation(X_v3, y, names_v3, "v3 (9 features, + spend_capped)")

```
--- Single 75/25 split (Ch067-070 baseline) ---
v3 (9 features), single split          (9 features): train_acc=1.000, test_acc=0.933

--- StratifiedKFold(5) cross-validation ---
v2 (8 features, Ch069 RFE)             (8 features): fold scores=[0.933 0.933 0.95  0.933 0.917], mean=0.933 +/- std=0.011
v3 (9 features, + spend_capped)        (9 features): fold scores=[0.933 0.933 0.95  0.933 0.917], mean=0.933 +/- std=0.011
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Toy data: 500 rows, 95/5 class split
rng = np.random.default_rng(0)
X = rng.normal(size=(500, 3))
y = np.array([0] * 475 + [1] * 25)

# First split off the test set (15% of 500)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=0, stratify=y
)
# Then split the remaining 425 rows into train (70/85 of it) and val (15/85 of it)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=15/85, random_state=0, stratify=y_train_full
)

print(f"Train: {len(X_train)} ({len(X_train)/500:.1%})")
print(f"Val:   {len(X_val)} ({len(X_val)/500:.1%})")
print(f"Test:  {len(X_test)} ({len(X_test)/500:.1%})")

```
Train: 350 (70.0%)
Val:   75 (15.0%)
Test:  75 (15.0%)
```

---

# Chapter 72: Metrics — Accuracy, Precision, Recall, F1, AUC-ROC, Log-Loss, MAE, and MSE

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy scikit-learn

### 2.1 Confusion matrix, precision, recall, and F1 on a 90/10 imbalanced toy

In [ ]:
import numpy as np
from sklearn.metrics import (confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score)

y_true = np.array([0]*90 + [1]*10)

# Model A: majority-class baseline -- predicts 0 for everyone
pred_a = np.zeros(100, dtype=int)
print("Model A (majority baseline):")
print(f"  Accuracy:  {accuracy_score(y_true, pred_a):.3f}")
print(f"  Precision: {precision_score(y_true, pred_a, zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y_true, pred_a, zero_division=0):.3f}")
print(f"  F1:        {f1_score(y_true, pred_a, zero_division=0):.3f}")
print(f"  Confusion matrix [[TN FP] [FN TP]]:\n{confusion_matrix(y_true, pred_a)}")
print()

# Model B: catches 8 of 10 positives, with 5 false alarms among the 90 negatives
pred_b = np.zeros(100, dtype=int)
pred_b[90:100] = [1,1,1,1,1,1,1,1,0,0]   # 8 of the 10 positives caught
pred_b[0:5] = 1                          # 5 false positives
print("Model B (catches most positives, a few false alarms):")
print(f"  Accuracy:  {accuracy_score(y_true, pred_b):.3f}")
print(f"  Precision: {precision_score(y_true, pred_b, zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y_true, pred_b, zero_division=0):.3f}")
print(f"  F1:        {f1_score(y_true, pred_b, zero_division=0):.3f}")
print(f"  Confusion matrix [[TN FP] [FN TP]]:\n{confusion_matrix(y_true, pred_b)}")

```
Model A (majority baseline):
  Accuracy:  0.900
  Precision: 0.000
  Recall:    0.000
  F1:        0.000
  Confusion matrix [[TN FP] [FN TP]]:
[[90  0]
 [10  0]]

Model B (catches most positives, a few false alarms):
  Accuracy:  0.930
  Precision: 0.615
  Recall:    0.800
  F1:        0.696
  Confusion matrix [[TN FP] [FN TP]]:
[[85  5]
 [ 2  8]]
```

### 2.2 ROC-AUC: ranking quality vs a single threshold's accuracy

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

y_true = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
proba_c = np.array([0.05, 0.10, 0.20, 0.30, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95])
proba_d = np.array([0.00, 0.00, 0.10, 0.20, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85])

for name, proba in [("Model C (calibrated)", proba_c),
                     ("Model D (shifted scores, same ranking)", proba_d)]:
    pred = (proba >= 0.5).astype(int)
    print(f"{name}:")
    print(f"  Predictions @ 0.5:         {pred.tolist()}")
    print(f"  Accuracy at threshold 0.5: {accuracy_score(y_true, pred):.3f}")
    print(f"  ROC-AUC:                   {roc_auc_score(y_true, proba):.3f}")

```
Model C (calibrated):
  Predictions @ 0.5:         [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
  Accuracy at threshold 0.5: 1.000
  ROC-AUC:                   1.000
Model D (shifted scores, same ranking):
  Predictions @ 0.5:         [0, 0, 0, 0, 0, 0, 1, 1, 1, 1]
  Accuracy at threshold 0.5: 0.900
  ROC-AUC:                   1.000
```

### 2.3 Log-loss: confidence vs correctness

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, log_loss

y_true = np.array([0, 0, 0, 1, 1, 1])
proba_e = np.array([0.20, 0.30, 0.10, 0.70, 0.80, 0.60])  # modest, all correct
proba_f = np.array([0.05, 0.02, 0.03, 0.98, 0.97, 0.01])  # confident, one miss

for name, proba in [("Model E (modest confidence, all correct)", proba_e),
                     ("Model F (overconfident, one miss)", proba_f)]:
    pred = (proba >= 0.5).astype(int)
    print(f"{name}:")
    print(f"  Predictions @ 0.5: {pred.tolist()}  (true: {y_true.tolist()})")
    print(f"  Accuracy:          {accuracy_score(y_true, pred):.3f}")
    print(f"  Log-loss:          {log_loss(y_true, proba):.3f}")

```
Model E (modest confidence, all correct):
  Predictions @ 0.5: [0, 0, 0, 1, 1, 1]  (true: [0, 0, 0, 1, 1, 1])
  Accuracy:          1.000
  Log-loss:          0.296
Model F (overconfident, one miss):
  Predictions @ 0.5: [0, 0, 0, 1, 1, 0]  (true: [0, 0, 0, 1, 1, 1])
  Accuracy:          0.833
  Log-loss:          0.793
```

### 2.4 Regression metrics: MAE vs MSE vs RMSE

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_true = np.array([10, 20, 30, 40, 50])
pred_g = np.array([11, 19, 31, 39, 51])  # off by 1, every time
pred_h = np.array([10, 20, 30, 40, 80])  # perfect x4, one huge miss

for name, pred in [("Model G (small, even errors)", pred_g),
                    ("Model H (one huge outlier)", pred_h)]:
    mae = mean_absolute_error(y_true, pred)
    mse = mean_squared_error(y_true, pred)
    print(f"{name}:")
    print(f"  Predictions: {pred.tolist()}  (true: {y_true.tolist()})")
    print(f"  MAE:  {mae:.3f}")
    print(f"  MSE:  {mse:.3f}")
    print(f"  RMSE: {np.sqrt(mse):.3f}")

```
Model G (small, even errors):
  Predictions: [11, 19, 31, 39, 51]  (true: [10, 20, 30, 40, 50])
  MAE:  1.000
  MSE:  1.000
  RMSE: 1.000
Model H (one huge outlier):
  Predictions: [10, 20, 30, 40, 80]  (true: [10, 20, 30, 40, 50])
  MAE:  6.000
  MSE:  180.000
  RMSE: 13.416
```

## 3. CinemaStream in Practice

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, log_loss, classification_report)

from cinemastream.ml.churn.train import build_churn_dataset, RANDOM_SEED
from cinemastream.ml.churn.data_quality import (inject_billing_data, impute_spend,
    detect_and_cap_outliers, prepare_features_v3)

churn_df = build_churn_dataset()
billed_df = inject_billing_data(churn_df)
imputed_df, _ = impute_spend(billed_df)
capped_df, _, _, _ = detect_and_cap_outliers(imputed_df)
X_v2, X_v3, y, names_v2, names_v3 = prepare_features_v3(capped_df)

X_train, X_test, y_train, y_test = train_test_split(
    X_v3, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)
print(f"Test set: {len(y_test)} rows, {y_test.sum()} churned ({y_test.mean():.1%})")
print()

# Majority-class baseline
y_pred_baseline = np.zeros_like(y_test)
print("--- Majority-class baseline ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_baseline):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_baseline, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_baseline, zero_division=0):.3f}")
print(f"F1:        {f1_score(y_test, y_pred_baseline, zero_division=0):.3f}")
print(f"Confusion matrix [[TN FP] [FN TP]]:\n{confusion_matrix(y_test, y_pred_baseline)}")
print()

# Random Forest (v3, 9 features) -- the Ch067-071 model
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("--- Random Forest (v3, 9 features) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf, zero_division=0):.3f}")
print(f"F1:        {f1_score(y_test, y_pred_rf, zero_division=0):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_rf):.3f}")
print(f"Log-loss:  {log_loss(y_test, y_proba_rf):.3f}")
print(f"Confusion matrix [[TN FP] [FN TP]]:\n{confusion_matrix(y_test, y_pred_rf)}")

```
Test set: 75 rows, 5 churned (6.7%)

--- Majority-class baseline ---
Accuracy:  0.933
Precision: 0.000
Recall:    0.000
F1:        0.000
Confusion matrix [[TN FP] [FN TP]]:
[[70  0]
 [ 5  0]]

--- Random Forest (v3, 9 features) ---
Accuracy:  0.933
Precision: 0.000
Recall:    0.000
F1:        0.000
ROC-AUC:   0.690
Log-loss:  0.249
Confusion matrix [[TN FP] [FN TP]]:
[[70  0]
 [ 5  0]]
```

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

from cinemastream.ml.churn.train import build_churn_dataset, RANDOM_SEED
from cinemastream.ml.churn.data_quality import (inject_billing_data, impute_spend,
    detect_and_cap_outliers, prepare_features_v3)

churn_df = build_churn_dataset()
billed_df = inject_billing_data(churn_df)
imputed_df, _ = impute_spend(billed_df)
capped_df, _, _, _ = detect_and_cap_outliers(imputed_df)
X_v2, X_v3, y, names_v2, names_v3 = prepare_features_v3(capped_df)

X_train, X_test, y_train, y_test = train_test_split(
    X_v3, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
rf.fit(X_train, y_train)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Threshold sweep:")
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    pred_t = (y_proba_rf >= t).astype(int)
    print(f"  threshold={t:.1f}: "
          f"precision={precision_score(y_test, pred_t, zero_division=0):.3f}, "
          f"recall={recall_score(y_test, pred_t, zero_division=0):.3f}, "
          f"f1={f1_score(y_test, pred_t, zero_division=0):.3f}, "
          f"accuracy={accuracy_score(y_test, pred_t):.3f}")

```
Threshold sweep:
  threshold=0.5: precision=0.000, recall=0.000, f1=0.000, accuracy=0.933
  threshold=0.4: precision=0.000, recall=0.000, f1=0.000, accuracy=0.920
  threshold=0.3: precision=0.200, recall=0.200, f1=0.200, accuracy=0.893
  threshold=0.2: precision=0.143, recall=0.200, f1=0.167, accuracy=0.867
  threshold=0.1: precision=0.154, recall=0.400, f1=0.222, accuracy=0.813
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import numpy as np
from sklearn.metrics import (confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score)

y_true = np.array([0]*180 + [1]*20)
# Build predictions matching the given confusion matrix:
# TN=175, FP=5 (among the 180 negatives); FN=15, TP=5 (among the 20 positives)
pred = np.zeros(200, dtype=int)
pred[0:5] = 1            # 5 false positives among the negatives
pred[180:185] = 1        # 5 true positives among the positives (rows 180-184)
                          # rows 185-199 (15 positives) stay predicted 0 -> FN

print(f"Accuracy:  {accuracy_score(y_true, pred):.3f}")
print(f"Precision: {precision_score(y_true, pred):.3f}")
print(f"Recall:    {recall_score(y_true, pred):.3f}")
print(f"F1:        {f1_score(y_true, pred):.3f}")
print(f"Confusion matrix:\n{confusion_matrix(y_true, pred)}")

# Majority-class baseline for comparison
pred_baseline = np.zeros(200, dtype=int)
print(f"\nBaseline accuracy: {accuracy_score(y_true, pred_baseline):.3f}")

```
Accuracy:  0.900
Precision: 0.500
Recall:    0.250
F1:        0.333
Confusion matrix:
[[175   5]
 [ 15   5]]

Baseline accuracy: 0.900
```

---

# Chapter 72a: Causal Thinking for Data & ML Engineers

## 0. Where You Are

## 1. The Concept

```
DiD = (treatment_post − treatment_pre) − (control_post − control_pre)
```

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas

### 2.1 Simpson's Paradox: the aggregate that lies

In [ ]:
import pandas as pd

# Simulated CTR experiment across two markets
# Treatment = new ad creative; Control = original creative
rows = [
    # market,           variant,      impressions, clicks
    ('Small market',    'control',         200,       80),  # 40% CTR
    ('Small market',    'treatment',      1800,      630),  # 35% CTR
    ('Large market',    'control',        1800,      360),  # 20% CTR
    ('Large market',    'treatment',       200,       34),  # 17% CTR
]
df = pd.DataFrame(rows, columns=['market', 'variant', 'impressions', 'clicks'])
df['ctr'] = df['clicks'] / df['impressions']

print("=== Per-Market CTR ===")
pivot = df.pivot_table(values='ctr', index='market',
                        columns='variant', aggfunc='first')
print(pivot.round(3).to_string())
print()

# Aggregate — collapse market
agg = df.groupby('variant').agg(
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum')
).reset_index()
agg['ctr'] = agg['clicks'] / agg['impressions']
print("=== Aggregate (market collapsed) ===")
for _, row in agg.iterrows():
    print(f"  {row['variant']:10s}: {row['clicks']:.0f}/{row['impressions']:.0f}"
          f" = {row['ctr']:.1%}")
print()
print("Within every market: treatment CTR < control CTR.")
print("Aggregate:           treatment CTR > control CTR — the REVERSAL.")
print()
print("Why: treatment was deployed mostly in the small market (40% base CTR).")
print("     Control dominated the large market (20% base CTR).")
print("     The aggregate rewards whichever variant landed in the better market.")

### 2.2 Confounders: the variable that drove both

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 500

# Confounder: engagement level (0 = casual, 1 = heavy user)
# Causes both plan price (heavy users upgrade) and watch hours
engagement = rng.integers(0, 2, size=n)

# Plan price: casual ≈ $8, heavy ≈ $18 (premium)
plan_price = 8 + engagement * 10 + rng.normal(0, 1.5, n)
plan_price = np.clip(plan_price, 4, 25)

# Watch hours per week: casual ≈ 5h, heavy ≈ 25h
watch_hours = 5 + engagement * 20 + rng.normal(0, 4, n)
watch_hours = np.clip(watch_hours, 0, None)

naive_corr = np.corrcoef(plan_price, watch_hours)[0, 1]
print(f"Naive correlation(plan_price, watch_hours):  r = {naive_corr:.3f}")
print("  → Looks like paying more CAUSES more watching!")
print()

for level, label in [(0, 'casual users'), (1, 'heavy users')]:
    mask = engagement == level
    r = np.corrcoef(plan_price[mask], watch_hours[mask])[0, 1]
    n_group = mask.sum()
    print(f"  Within {label} (n={n_group}):  r = {r:.3f}")

print()
print("Within each engagement level: correlation nearly vanishes.")
print("Engagement drives both — conditioning on it breaks the spurious link.")

### 2.3 Causal DAG: drawing the story

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    CDN["Regional CDN quality<br/>(infrastructure)"]
    ENG["User engagement<br/>(watch hours, binge patterns)"]
    PLAN["Plan tier<br/>(Basic / Premium)"]
    BUF["Buffering / lag<br/>(playback issues)"]
    CHURN["Churn"]

    ENG -->|causes| PLAN
    ENG -->|causes| BUF
    CDN -->|causes| BUF
    BUF -->|causes| CHURN

    PLAN -.->|no direct causal path —<br/>correlation flows back<br/>through engagement| CHURN
</div>
"""))

### 2.4 Difference-in-Differences: isolating the CDN effect

In [ ]:
import pandas as pd

# Before/after CDN upgrade — HD/4K vs SD users
# Parallel trends assumption: both groups had similar complaint trajectories pre-upgrade
did_df = pd.DataFrame({
    'group':   ['HD/4K users', 'HD/4K users', 'SD users', 'SD users'],
    'period':  ['Q4 (pre)',    'Q1 (post)',   'Q4 (pre)', 'Q1 (post)'],
    'buffering_per_100': [18, 8, 12, 10],
})
print(did_df.to_string(index=False))
print()

pre_treat  = 18
post_treat = 8
pre_ctrl   = 12
post_ctrl  = 10

delta_treat = post_treat - pre_treat
delta_ctrl  = post_ctrl  - pre_ctrl
did         = delta_treat - delta_ctrl

print(f"HD/4K users change: {pre_treat} → {post_treat}  (Δ = {delta_treat:+d})")
print(f"SD users change:    {pre_ctrl} → {post_ctrl}  (Δ = {delta_ctrl:+d})")
print(f"\nDiD estimate: ({delta_treat}) − ({delta_ctrl}) = {did:+d} per 100 users")
print()
print("SD users fell by 2 — background trend (seasonality, account maturation).")
print("HD/4K users fell by 10. Net causal estimate: 10 − 2 = −8 per 100.")
print("The CDN upgrade causally reduced HD/4K buffering complaints by ~8 per 100,")
print("over and above whatever would have changed anyway.")

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd

# CinemaStream Q3 — churn by region and plan
# VN market: predominantly Basic (legacy entry pricing, large user base)
# SG market: predominantly Premium (higher disposable income, smaller user base)
cs_data = {
    'region':  ['VN',     'VN',       'SG',    'SG'],
    'plan':    ['Basic',  'Premium',  'Basic', 'Premium'],
    'users':   [450,       50,         50,      450],
    'churned': [90,        15,          5,       68],
}
df = pd.DataFrame(cs_data)
df['churn_rate'] = df['churned'] / df['users']

print("CinemaStream Q3 — churn by region and plan")
print("=" * 55)
for _, row in df.iterrows():
    print(f"  {row['region']}  {row['plan']:8s}:  "
          f"{row['churned']:3.0f}/{row['users']:3.0f} churned  "
          f"= {row['churn_rate']:.1%}")

print()
agg = df.groupby('plan').agg(
    users=('users', 'sum'),
    churned=('churned', 'sum')
).reset_index()
agg['churn_rate'] = agg['churned'] / agg['users']
print("Aggregate (region ignored):")
for _, row in agg.iterrows():
    print(f"  {row['plan']:8s}: {row['churned']:.0f}/{row['users']:.0f} "
          f"= {row['churn_rate']:.1%}")

print()
print("Within each region: Premium churn EXCEEDS Basic churn.")
print("Aggregate: Premium churn < Basic churn — REVERSAL.")
print("Confounder: region. Premium is concentrated in SG (lower base rate).")
print("Conclusion: do NOT present this aggregate as 'Premium is stickier.'")

```
engagement level ──→ plan tier (Basic/Premium)
engagement level ──→ watch hours/week
engagement level ──→ content type (SD vs HD/4K)
CDN quality      ──→ buffering events
content type     ──→ buffering demand (HD needs more bandwidth)
buffering events ──→ churn
plan tier: no direct arrow to churn
```

In [ ]:
import pandas as pd

# Vietnam DiD: Q2 (pre CDN upgrade) vs Q3 (post CDN upgrade)
# Treatment: Premium plan users in VN (heavier HD/4K — more bandwidth-sensitive)
# Control:   Basic plan users in VN (lighter streaming load)
# Parallel trends assumption: both groups trended similarly in Q1→Q2 (pre-upgrade)

vn_did = pd.DataFrame({
    'plan':       ['Premium', 'Premium',   'Basic',   'Basic'],
    'quarter':    ['Q2 (pre)', 'Q3 (post)', 'Q2 (pre)', 'Q3 (post)'],
    'churn_rate': [0.36,        0.30,        0.22,        0.20],
})
print("Vietnam — churn before/after CDN upgrade")
print(vn_did.to_string(index=False))
print()

pre_prem  = 0.36
post_prem = 0.30
pre_basic = 0.22
post_basic = 0.20

delta_prem  = post_prem  - pre_prem
delta_basic = post_basic - pre_basic
did         = delta_prem - delta_basic

print(f"Premium: {pre_prem:.0%} → {post_prem:.0%}  (Δ = {delta_prem:+.0%})")
print(f"Basic:   {pre_basic:.0%} → {post_basic:.0%}  (Δ = {delta_basic:+.0%})")
print(f"\nDiD estimate: {delta_prem:+.0%} − {delta_basic:+.0%} = {did:+.0%}")
print()
print("Background trend (Basic drop): −2pp.")
print("Premium dropped an extra −4pp beyond background.")
print("CDN upgrade appears to have causally reduced Premium churn in VN by ~4pp.")
print("This is consistent with the DAG: CDN quality → buffering → churn.")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import pandas as pd

rows = [
    ('US',     'A', 1000, 200),
    ('US',     'B', 1000, 150),
    ('Canada', 'A',  100,  40),
    ('Canada', 'B', 1000, 380),
]
df = pd.DataFrame(rows, columns=['country', 'layout', 'visitors', 'conversions'])
df['rate'] = df['conversions'] / df['visitors']

print("Per country:")
for _, r in df.iterrows():
    print(f"  {r['country']:7s} Layout {r['layout']}: {r['rate']:.1%}")

agg = df.groupby('layout').agg(v=('visitors','sum'), c=('conversions','sum')).reset_index()
agg['rate'] = agg['c'] / agg['v']
print("\nAggregate:")
for _, r in agg.iterrows():
    print(f"  Layout {r['layout']}: {r['c']:.0f}/{r['v']:.0f} = {r['rate']:.1%}")

In [ ]:
pre_treat   = 0.30
post_treat  = 0.25
pre_ctrl    = 0.20
post_ctrl   = 0.19

delta_treat = post_treat - pre_treat   # -5pp
delta_ctrl  = post_ctrl  - pre_ctrl    # -1pp
did         = delta_treat - delta_ctrl # -4pp

print(f"Premium (treatment): {pre_treat:.0%} → {post_treat:.0%}  Δ = {delta_treat:+.0%}")
print(f"Basic (control):     {pre_ctrl:.0%} → {post_ctrl:.0%}  Δ = {delta_ctrl:+.0%}")
print(f"DiD estimate: {delta_treat:+.0%} − {delta_ctrl:+.0%} = {did:+.0%}")
print()
print("Causal estimate: quality upgrade reduced Premium churn by ~4pp,")
print("over and above the 1pp background trend.")

---

# Chapter 73: Hyperparameter Tuning — Grid Search, Random Search, and Bayesian Optimization with Optuna

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy optuna pandas scikit-learn

### 2.1 Parameters vs hyperparameters, and the manual loop

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(n_samples=600, n_features=10, n_informative=4,
                           random_state=7)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=7, stratify=y_trainval)

print(f"train={len(y_train)}, validation={len(y_val)}, test={len(y_test)}")
for depth in [1, 2, 3, 5, 8, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=7)
    tree.fit(X_train, y_train)
    print(f"max_depth={str(depth):>4}: "
          f"train acc={tree.score(X_train, y_train):.3f}, "
          f"validation acc={tree.score(X_val, y_val):.3f}")

```
train=337, validation=113, test=150
max_depth=   1: train acc=0.864, validation acc=0.823
max_depth=   2: train acc=0.890, validation acc=0.823
max_depth=   3: train acc=0.917, validation acc=0.841
max_depth=   5: train acc=0.970, validation acc=0.841
max_depth=   8: train acc=0.997, validation acc=0.788
max_depth=None: train acc=1.000, validation acc=0.788
```

In [ ]:
best_tree = DecisionTreeClassifier(max_depth=3, random_state=7)
best_tree.fit(X_train, y_train)
print(f"Chosen: max_depth=3 (tied with 5 -> prefer simpler).")
print(f"Final, single look at the test set: {best_tree.score(X_test, y_test):.3f}")

```
Chosen: max_depth=3 (tied with 5 -> prefer simpler).
Final, single look at the test set: 0.873
```

### 2.2 Grid search: exhaustive, simple, exponential

In [ ]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
param_grid = {"C": [0.1, 1, 10], "gamma": [0.01, 0.1, 1]}

search = GridSearchCV(SVC(random_state=7), param_grid, cv=cv)
search.fit(X_trainval, y_trainval)

print(f"Candidates tried: {len(search.cv_results_['params'])}")
print(f"Best params: {search.best_params_}")
print(f"Best mean CV accuracy: {search.best_score_:.3f}")

results = pd.DataFrame(search.cv_results_)[
    ["param_C", "param_gamma", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")
print(results.head(4).to_string(index=False))

print(f"Test accuracy of best model (refit on all train+val): "
      f"{search.score(X_test, y_test):.3f}")

```
Candidates tried: 9
Best params: {'C': 1, 'gamma': 0.1}
Best mean CV accuracy: 0.924
 param_C  param_gamma  mean_test_score  std_test_score  rank_test_score
     1.0         0.10         0.924444        0.016330                1
    10.0         0.01         0.908889        0.019116                2
     0.1         0.10         0.893333        0.015072                3
    10.0         0.10         0.891111        0.030144                4
Test accuracy of best model (refit on all train+val): 0.913
```

### 2.3 Random search: better coverage on the same budget

In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

space = {"C": loguniform(1e-2, 1e2), "gamma": loguniform(1e-3, 1e1)}
rand_search = RandomizedSearchCV(SVC(random_state=7), space, n_iter=9,
                                 cv=cv, random_state=7)
rand_search.fit(X_trainval, y_trainval)

print(f"Candidates tried: {len(rand_search.cv_results_['params'])}")
print(f"Best params: C={rand_search.best_params_['C']:.4f}, "
      f"gamma={rand_search.best_params_['gamma']:.4f}")
print(f"Best mean CV accuracy: {rand_search.best_score_:.3f}")
print(f"Test accuracy of best model: {rand_search.score(X_test, y_test):.3f}")

```
Candidates tried: 9
Best params: C=0.1185, gamma=0.0999
Best mean CV accuracy: 0.898
Test accuracy of best model: 0.887
```

### 2.4 Bayesian optimization: Optuna and the TPE sampler

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    x = trial.suggest_float("x", -10, 10)
    return (x - 2) ** 2

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=7))
study.optimize(objective, n_trials=30)

print(f"Best x: {study.best_params['x']:.4f}  (true optimum: x=2)")
print(f"Best objective value: {study.best_value:.6f}")
print("First 3 trials (exploring):")
for t in study.trials[:3]:
    print(f"  trial {t.number}: x={t.params['x']:8.4f} -> objective={t.value:.4f}")
print("Last 3 trials (exploiting):")
for t in study.trials[-3:]:
    print(f"  trial {t.number}: x={t.params['x']:8.4f} -> objective={t.value:.4f}")

```
Best x: 2.1667  (true optimum: x=2)
Best objective value: 0.027799
First 3 trials (exploring):
  trial 0: x= -8.4738 -> objective=109.7012
  trial 1: x=  5.5984 -> objective=12.9483
  trial 2: x= -1.2318 -> objective=10.4446
Last 3 trials (exploiting):
  trial 27: x=  6.3759 -> objective=19.1488
  trial 28: x=  4.1436 -> objective=4.5950
  trial 29: x=  1.3419 -> objective=0.4331
```

### 2.5 Where do tuning decisions spend their data?

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    A["All labelled data (300 rows)"] --> B["Training set (225 rows, 16 churned)"]
    A --> C["Test set (75 rows, 5 churned) — locked away"]
    B --> D["StratifiedKFold(5) cross-validation"]
    D --> E["Score every hyperparameter candidate (scoring = F1)"]
    E --> F["Select winner by mean CV F1"]
    B --> G["Out-of-fold probabilities choose the decision threshold"]
    F --> H["Refit winner on the full training set"]
    G --> H
    H --> I["ONE final evaluation on the test set"]
    C --> I
</div>
"""))

## 3. CinemaStream in Practice

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_score, train_test_split)

from cinemastream.ml.churn.train import build_churn_dataset, RANDOM_SEED
from cinemastream.ml.churn.data_quality import (inject_billing_data, impute_spend,
    detect_and_cap_outliers, prepare_features_v3)

churn_df = build_churn_dataset()
billed_df = inject_billing_data(churn_df)
imputed_df, _ = impute_spend(billed_df)
capped_df, _, _, _ = detect_and_cap_outliers(imputed_df)
X_v2, X_v3, y, names_v2, names_v3 = prepare_features_v3(capped_df)
X_train, X_test, y_train, y_test = train_test_split(
    X_v3, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y)
print(f"Train: {len(y_train)} rows ({y_train.sum()} churned), "
      f"Test: {len(y_test)} rows ({y_test.sum()} churned)")

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
PARAM_GRID = {
    "class_weight": [None, "balanced", {0: 1, 1: 5}, {0: 1, 1: 10}],
    "max_depth": [None, 3, 5, 8],
    "min_samples_leaf": [1, 2, 4],
}

rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
grid_acc = GridSearchCV(rf, PARAM_GRID, scoring="accuracy", cv=CV)
grid_acc.fit(X_train, y_train)
print(f"Best params: {grid_acc.best_params_}")
print(f"Best CV accuracy: {grid_acc.best_score_:.3f}")
rec = cross_val_score(grid_acc.best_estimator_, X_train, y_train,
                      scoring="recall", cv=CV)
print(f"That 'best' model's CV recall: {np.round(rec, 3)} (mean {rec.mean():.3f})")

```
Train: 225 rows (16 churned), Test: 75 rows (5 churned)
Best params: {'class_weight': None, 'max_depth': None, 'min_samples_leaf': 2}
Best CV accuracy: 0.929
That 'best' model's CV recall: [0. 0. 0. 0. 0.] (mean 0.000)
```

In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV
import optuna

# -- Grid search, same 48 candidates, scored by F1 this time
grid_f1 = GridSearchCV(rf, PARAM_GRID, scoring="f1", cv=CV)
grid_f1.fit(X_train, y_train)
print(f"Grid   best CV F1: {grid_f1.best_score_:.3f}  {grid_f1.best_params_}")

# -- Random search: wider space, 30-candidate budget
space = {
    "n_estimators": randint(50, 401),
    "max_depth": [None, 3, 5, 8, 12],
    "min_samples_leaf": randint(1, 9),
    "min_samples_split": randint(2, 11),
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced", {0: 1, 1: 5}, {0: 1, 1: 10}],
}
rand_f1 = RandomizedSearchCV(RandomForestClassifier(random_state=RANDOM_SEED),
                             space, n_iter=30, scoring="f1", cv=CV,
                             random_state=RANDOM_SEED)
rand_f1.fit(X_train, y_train)
print(f"Random best CV F1: {rand_f1.best_score_:.3f}  {rand_f1.best_params_}")

# -- Optuna TPE: same space, 40 trials (suggest_categorical can't hold
#    dicts, so class_weight rides along as a string key)
cw_options = {"none": None, "balanced": "balanced",
              "w5": {0: 1, 1: 5}, "w10": {0: 1, 1: 10}}

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_categorical("max_depth", [None, 3, 5, 8, 12]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "max_features": trial.suggest_categorical("max_features",
                                                  ["sqrt", "log2", None]),
    }
    cw_key = trial.suggest_categorical("class_weight", list(cw_options))
    model = RandomForestClassifier(**params, class_weight=cw_options[cw_key],
                                   random_state=RANDOM_SEED)
    return cross_val_score(model, X_train, y_train, scoring="f1", cv=CV).mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=40)
print(f"Optuna best CV F1: {study.best_value:.3f}  {study.best_params}")

```
Grid   best CV F1: 0.266  {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 2}
Random best CV F1: 0.282  {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 8, 'min_samples_split': 5, 'n_estimators': 51}
Optuna best CV F1: 0.300  {'n_estimators': 364, 'max_depth': 3, 'min_samples_leaf': 6, 'min_samples_split': 9, 'max_features': 'log2', 'class_weight': 'balanced'}
```

In [ ]:
from sklearn.model_selection import cross_val_predict
from cinemastream.ml.churn.metrics import threshold_sweep

final_model = RandomForestClassifier(
    n_estimators=389, max_depth=5, min_samples_leaf=7, min_samples_split=7,
    max_features=None, class_weight="balanced", random_state=RANDOM_SEED)

proba_oof = cross_val_predict(final_model, X_train, y_train,
                              cv=CV, method="predict_proba")[:, 1]
print("Out-of-fold (training) threshold sweep:")
threshold_sweep(y_train, proba_oof)

```
Out-of-fold (training) threshold sweep:
Threshold sweep:
  threshold=0.5: precision=0.149, recall=0.438, f1=0.222, accuracy=0.782
  threshold=0.4: precision=0.154, recall=0.625, f1=0.247, accuracy=0.729
  threshold=0.3: precision=0.128, recall=0.688, f1=0.216, accuracy=0.644
  threshold=0.2: precision=0.113, recall=0.812, f1=0.198, accuracy=0.533
  threshold=0.1: precision=0.085, recall=0.812, f1=0.154, accuracy=0.364
```

In [ ]:
from cinemastream.ml.churn.metrics import evaluate_classifier

final_model.fit(X_train, y_train)
proba_test = final_model.predict_proba(X_test)[:, 1]

evaluate_classifier(y_test, (proba_test >= 0.3).astype(int), proba_test,
                    label="Tuned RF @ threshold 0.3")
print()
evaluate_classifier(y_test, (proba_test >= 0.5).astype(int),
                    label="Tuned RF @ default threshold 0.5 (for comparison)")

```
--- Tuned RF @ threshold 0.3 ---
Accuracy:  0.733
Precision: 0.174
Recall:    0.800
F1:        0.286
ROC-AUC:   0.849
Log-loss:  0.397
Confusion matrix [[TN FP] [FN TP]]:
[[51 19]
 [ 1  4]]

--- Tuned RF @ default threshold 0.5 (for comparison) ---
Accuracy:  0.840
Precision: 0.231
Recall:    0.600
F1:        0.333
Confusion matrix [[TN FP] [FN TP]]:
[[60 10]
 [ 2  3]]
```

In [ ]:
from sklearn.metrics import f1_score, make_scorer, recall_score

recall_at_03 = make_scorer(
    lambda yt, yp: recall_score(yt, (yp >= 0.3).astype(int), zero_division=0),
    response_method="predict_proba")
f1_at_03 = make_scorer(
    lambda yt, yp: f1_score(yt, (yp >= 0.3).astype(int), zero_division=0),
    response_method="predict_proba")

X_full = np.vstack([X_train, X_test])
y_full = np.concatenate([y_train, y_test])
acc = cross_val_score(final_model, X_full, y_full, cv=CV)
rec = cross_val_score(final_model, X_full, y_full, scoring=recall_at_03, cv=CV)
f1 = cross_val_score(final_model, X_full, y_full, scoring=f1_at_03, cv=CV)
print(f"accuracy (at 0.5): fold scores={np.round(acc, 3)}, "
      f"mean={acc.mean():.3f} +/- std={acc.std():.3f}")
print(f"recall @ 0.3:      fold scores={np.round(rec, 3)}, "
      f"mean={rec.mean():.3f} +/- std={rec.std():.3f}")
print(f"f1 @ 0.3:          fold scores={np.round(f1, 3)}, "
      f"mean={f1.mean():.3f} +/- std={f1.std():.3f}")

```
accuracy (at 0.5): fold scores=[0.85  0.8   0.767 0.783 0.817], mean=0.803 +/- std=0.029
recall @ 0.3:      fold scores=[0.75 0.75 0.75 1.   0.6 ], mean=0.770 +/- std=0.129
f1 @ 0.3:          fold scores=[0.353 0.222 0.24  0.258 0.214], mean=0.258 +/- std=0.050
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 74: Model Comparison and the Bias-Variance Trade-off — Under/Overfitting and Learning Curves

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy scikit-learn

### 2.1 Seeing both failure modes in one toy

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(7)
X = np.sort(rng.uniform(0, 1, 40))[:, None]
y = np.sin(2 * np.pi * X).ravel() + rng.normal(0, 0.2, 40)

X_train, X_test = X[::2], X[1::2]   # every other point: 20 train, 20 test
y_train, y_test = y[::2], y[1::2]

for degree in [1, 4, 15]:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)
    mse_train = mean_squared_error(y_train, model.predict(X_train))
    mse_test = mean_squared_error(y_test, model.predict(X_test))
    print(f"degree={degree:>2}: train MSE={mse_train:.3f}, test MSE={mse_test:.3f}")

```
degree= 1: train MSE=0.303, test MSE=0.291
degree= 4: train MSE=0.041, test MSE=0.035
degree=15: train MSE=0.023, test MSE=0.044
```

### 2.2 The validation curve: score vs complexity

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, validation_curve
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(n_samples=600, n_features=10, n_informative=4,
                           random_state=7)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
depths = [1, 2, 3, 4, 6, 8, 10, 12]

train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=7), X, y,
    param_name="max_depth", param_range=depths, cv=cv)

print(f"{'max_depth':>9s} {'train acc':>10s} {'CV acc':>16s}")
for d, tr, va in zip(depths, train_scores, val_scores):
    print(f"{d:>9d} {tr.mean():>10.3f} {va.mean():>10.3f} +/- {va.std():.3f}")

```
max_depth  train acc           CV acc
        1      0.855      0.842 +/- 0.014
        2      0.880      0.862 +/- 0.029
        3      0.905      0.877 +/- 0.028
        4      0.933      0.875 +/- 0.012
        6      0.972      0.878 +/- 0.025
        8      0.993      0.878 +/- 0.010
       10      0.999      0.872 +/- 0.017
       12      1.000      0.873 +/- 0.019
```

### 2.3 The learning curve: score vs data volume

In [ ]:
from sklearn.model_selection import learning_curve

for label, model in [
    ("High-bias model (decision stump, max_depth=1)",
     DecisionTreeClassifier(max_depth=1, random_state=7)),
    ("High-variance model (unconstrained tree)",
     DecisionTreeClassifier(random_state=7)),
]:
    sizes, tr, va = learning_curve(model, X, y, cv=cv,
                                   train_sizes=np.linspace(0.2, 1.0, 5),
                                   shuffle=True, random_state=7)
    print(f"{label}:")
    print(f"{'train rows':>10s} {'train acc':>10s} {'CV acc':>10s} {'gap':>7s}")
    for n, t, v in zip(sizes, tr, va):
        print(f"{n:>10d} {t.mean():>10.3f} {v.mean():>10.3f} {t.mean()-v.mean():>7.3f}")
    print()

```
High-bias model (decision stump, max_depth=1):
train rows  train acc     CV acc     gap
        96      0.850      0.828   0.022
       192      0.854      0.827   0.028
       288      0.849      0.835   0.014
       384      0.855      0.843   0.011
       480      0.855      0.842   0.013

High-variance model (unconstrained tree):
train rows  train acc     CV acc     gap
        96      1.000      0.803   0.197
       192      1.000      0.845   0.155
       288      1.000      0.867   0.133
       384      1.000      0.875   0.125
       480      1.000      0.873   0.127
```

### 2.4 Comparing models fairly: same folds, same metric, honest ties

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Logistic Regression (scaled)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=7))]),
    "Decision Tree (max_depth=3)": DecisionTreeClassifier(
        max_depth=3, random_state=7),
    "Random Forest (100 trees)": RandomForestClassifier(
        n_estimators=100, random_state=7),
}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv)  # same cv object = same folds
    print(f"{name:30s} CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}  "
          f"folds={np.round(scores, 3)}")

```
Logistic Regression (scaled)   CV accuracy: 0.863 +/- 0.016  folds=[0.842 0.858 0.892 0.867 0.858]
Decision Tree (max_depth=3)    CV accuracy: 0.877 +/- 0.028  folds=[0.883 0.833 0.908 0.9   0.858]
Random Forest (100 trees)      CV accuracy: 0.912 +/- 0.021  folds=[0.933 0.883 0.933 0.892 0.917]
```

## 3. CinemaStream in Practice

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from cinemastream.ml.churn.train import RANDOM_SEED
from cinemastream.ml.churn.hyperparameter_tuning import CV, load_split

X_train, X_test, y_train, y_test, names = load_split()
print(f"Comparison data: the {len(y_train)} training rows "
      f"({y_train.sum()} churned). Test set: untouched this chapter.")

TUNED_RF_PARAMS = dict(
    n_estimators=389, max_depth=5, min_samples_leaf=7, min_samples_split=7,
    max_features=None, class_weight="balanced", random_state=RANDOM_SEED)

candidates = {
    "Tuned Random Forest (Ch073)":
        RandomForestClassifier(**TUNED_RF_PARAMS),
    "Logistic Regression (balanced, scaled)":
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(class_weight="balanced",
                                             max_iter=1000,
                                             random_state=RANDOM_SEED))]),
    "Decision Tree (depth=3, balanced)":
        DecisionTreeClassifier(max_depth=3, class_weight="balanced",
                               random_state=RANDOM_SEED),
    "Gradient Boosting (defaults)":
        GradientBoostingClassifier(random_state=RANDOM_SEED),
    "KNN (k=5, scaled)":
        Pipeline([("scaler", StandardScaler()),
                  ("clf", KNeighborsClassifier(n_neighbors=5))]),
}

print(f"{'Model':40s} {'CV ROC-AUC':>18s} {'CV F1 @ 0.5':>18s}")
for name, model in candidates.items():
    auc = cross_val_score(model, X_train, y_train, scoring="roc_auc", cv=CV)
    f1 = cross_val_score(model, X_train, y_train, scoring="f1", cv=CV)
    print(f"{name:40s} {auc.mean():>10.3f} +/- {auc.std():.3f} "
          f"{f1.mean():>10.3f} +/- {f1.std():.3f}")

```
Comparison data: the 225 training rows (16 churned). Test set: untouched this chapter.
Model                                            CV ROC-AUC        CV F1 @ 0.5
Tuned Random Forest (Ch073)                   0.733 +/- 0.123      0.226 +/- 0.086
Logistic Regression (balanced, scaled)        0.606 +/- 0.151      0.146 +/- 0.041
Decision Tree (depth=3, balanced)             0.617 +/- 0.138      0.282 +/- 0.175
Gradient Boosting (defaults)                  0.767 +/- 0.102      0.067 +/- 0.133
KNN (k=5, scaled)                             0.733 +/- 0.107      0.000 +/- 0.000
```

In [ ]:
from sklearn.metrics import roc_auc_score

probes = {
    "Unconstrained RF (Ch068 defaults)":
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
    "Tuned RF (Ch073, regularised)":
        RandomForestClassifier(**TUNED_RF_PARAMS),
    "Decision stump (max_depth=1, balanced)":
        DecisionTreeClassifier(max_depth=1, class_weight="balanced",
                               random_state=RANDOM_SEED),
}
print(f"{'Model':40s} {'train ROC-AUC':>14s} {'CV ROC-AUC':>18s} {'gap':>7s}")
for name, model in probes.items():
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    cv_auc = cross_val_score(model, X_train, y_train, scoring="roc_auc", cv=CV)
    print(f"{name:40s} {train_auc:>14.3f} {cv_auc.mean():>10.3f} +/- {cv_auc.std():.3f}"
          f" {train_auc - cv_auc.mean():>7.3f}")

```
Model                                     train ROC-AUC         CV ROC-AUC     gap
Unconstrained RF (Ch068 defaults)                 1.000      0.774 +/- 0.091   0.226
Tuned RF (Ch073, regularised)                     0.958      0.733 +/- 0.123   0.226
Decision stump (max_depth=1, balanced)            0.660      0.598 +/- 0.080   0.062
```

In [ ]:
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import learning_curve

X_full = np.vstack([X_train, X_test])
y_full = np.concatenate([y_train, y_test])

recall_at_03 = make_scorer(
    lambda yt, yp: recall_score(yt, (yp >= 0.3).astype(int), zero_division=0),
    response_method="predict_proba")

model = RandomForestClassifier(**TUNED_RF_PARAMS)
for scoring, label in [("roc_auc", "ROC-AUC"),
                       (recall_at_03, "recall @ threshold 0.3")]:
    sizes, tr, va = learning_curve(model, X_full, y_full, cv=CV,
                                   scoring=scoring,
                                   train_sizes=np.linspace(0.3, 1.0, 6),
                                   shuffle=True, random_state=RANDOM_SEED)
    print(f"Learning curve -- {label}:")
    print(f"{'train rows':>10s} {'train score':>12s} {'val score':>22s}")
    for n, t, v in zip(sizes, tr, va):
        print(f"{n:>10d} {t.mean():>12.3f} {v.mean():>12.3f} +/- {v.std():.3f}")
    print()

```
Learning curve -- ROC-AUC:
train rows  train score              val score
        72        0.961        0.620 +/- 0.172
       105        0.969        0.621 +/- 0.179
       139        0.979        0.624 +/- 0.144
       172        0.980        0.685 +/- 0.172
       206        0.971        0.752 +/- 0.102
       240        0.968        0.765 +/- 0.085

Learning curve -- recall @ threshold 0.3:
train rows  train score              val score
        72        1.000        0.560 +/- 0.299
       105        1.000        0.570 +/- 0.291
       139        1.000        0.590 +/- 0.269
       172        1.000        0.550 +/- 0.367
       206        1.000        0.670 +/- 0.189
       240        1.000        0.720 +/- 0.169
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

ks = [1, 3, 5, 15, 51, 151]
tr, va = validation_curve(KNeighborsClassifier(), X, y,
                          param_name="n_neighbors", param_range=ks, cv=cv)
print("   k  train acc     CV acc")
for k, t, v in zip(ks, tr, va):
    print(f"{k:>4d} {t.mean():>10.3f} {v.mean():>10.3f}")

```
   k  train acc     CV acc
   1      1.000      0.885
   3      0.946      0.905
   5      0.929      0.902
  15      0.917      0.898
  51      0.887      0.878
 151      0.853      0.848
```

---

# Chapter 74a: Unsupervised Learning — Clustering, PCA & Anomaly Detection

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas scikit-learn

### 2.1 K-means: elbow method and fit

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import numpy as np

# Three well-separated 2D blobs
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# Elbow: inertia for k = 1 to 7
print("Elbow method:")
print(f"{'k':>3}  {'inertia':>10}")
for k in range(1, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    marker = "  ← elbow" if k == 3 else ""
    print(f"{k:>3}  {km.inertia_:>10.1f}{marker}")

# Final fit with k=3
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km3.fit_predict(X)
print(f"\nCluster sizes: {np.bincount(labels).tolist()}")
print(f"Centroids:\n{km3.cluster_centers_.round(2)}")
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(y_true, labels)
print(f"\nAdjusted Rand Index vs true labels: {ari:.3f}  (1.0 = perfect)")

### 2.2 PCA: variance explained and loadings

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

rng = np.random.default_rng(42)
n = 200
F = rng.normal(0, 1, n)  # shared latent factor
X = np.column_stack([
    F,
    F + rng.normal(0, 0.3, n),   # near-duplicate of F
    F + rng.normal(0, 0.5, n),   # correlated with F
    rng.normal(0, 1, n),           # independent
    rng.normal(0, 1, n),           # independent
])
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(random_state=42)
pca.fit(X_scaled)

print(f"{'PC':>4}  {'Var explained':>14}  {'Cumulative':>10}")
cumul = 0.0
for i, v in enumerate(pca.explained_variance_ratio_, 1):
    cumul += v
    print(f"PC{i:>2}  {v:>10.3f} ({v:.1%})  {cumul:>9.1%}")

print("\nPC1 loadings (positive = feature aligned with latent factor):")
for i, load in enumerate(pca.components_[0], 1):
    print(f"  Feature {i}: {load:+.3f}")

### 2.3 IsolationForest: flagging outliers

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

rng = np.random.default_rng(42)
X_normal   = rng.normal(0, 1, (97, 2))
X_anomaly  = np.array([[10.0, 10.0], [12.0, 8.0], [11.0, 11.0]])
X = np.vstack([X_normal, X_anomaly])

iso = IsolationForest(contamination=0.03, random_state=42)
iso.fit(X)
preds = iso.predict(X)   # 1 = normal, -1 = anomaly

n_flagged  = (preds == -1).sum()
bots_caught = (preds[97:] == -1).sum()

print(f"Total points:            {len(X)}")
print(f"Flagged as anomalous:    {n_flagged}")
print(f"True anomalies detected: {bots_caught}/3")

scores = iso.score_samples(X)
print(f"\nAnomaly scores (lower = more isolated):")
print(f"  Normal (n=97):   mean = {scores[:97].mean():.3f}, "
      f"min = {scores[:97].min():.3f}")
print(f"  Anomalies (n=3): mean = {scores[97:].mean():.3f}, "
      f"max = {scores[97:].max():.3f}")

## 3. CinemaStream in Practice

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

# Synthetic CinemaStream pilot users: 300 rows, plan-driven behaviour distributions
# Ratios: Free 56% (168), Basic 28% (84), Premium 16% (48)
segments = [
    ('Free',    168, (5.0,  2.5), (2.0, 0.8), (8,  4)),
    ('Basic',    84, (12.0, 3.5), (4.0, 1.0), (15, 4)),
    ('Premium',  48, (22.0, 4.0), (6.0, 1.2), (22, 3)),
]
rows = []
for plan, n, (wh_mu, wh_sd), (ng_mu, ng_sd), (da_mu, da_sd) in segments:
    for _ in range(n):
        rows.append({
            'plan': plan,
            'watch_hours_week':  max(0.0, rng.normal(wh_mu, wh_sd)),
            'num_genres':        max(1,   round(rng.normal(ng_mu, ng_sd))),
            'days_active':       max(1,   round(rng.normal(da_mu, da_sd))),
        })
df = pd.DataFrame(rows)

features = ['watch_hours_week', 'num_genres', 'days_active']
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X_scaled)

print("CinemaStream user clusters (k=3, features scaled):")
for c in sorted(df['cluster'].unique()):
    grp = df[df['cluster'] == c]
    pd_str = {k: v for k, v in grp['plan'].value_counts().items()}
    print(f"\nCluster {c}  (n={len(grp)}):")
    print(f"  avg watch_hours_week  = {grp['watch_hours_week'].mean():.1f}")
    print(f"  avg num_genres        = {grp['num_genres'].mean():.1f}")
    print(f"  avg days_active       = {grp['days_active'].mean():.1f}")
    print(f"  plan breakdown        = {pd_str}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
segments = [
    ('Free',    168, (5.0,  2.5), (2.0, 0.8), (8,  4)),
    ('Basic',    84, (12.0, 3.5), (4.0, 1.0), (15, 4)),
    ('Premium',  48, (22.0, 4.0), (6.0, 1.2), (22, 3)),
]
rows = []
for plan, n, (wh_mu, wh_sd), (ng_mu, ng_sd), (da_mu, da_sd) in segments:
    for _ in range(n):
        rows.append({
            'plan': plan,
            'watch_hours_week':  max(0.0, rng.normal(wh_mu, wh_sd)),
            'num_genres':        max(1,   round(rng.normal(ng_mu, ng_sd))),
            'days_active':       max(1,   round(rng.normal(da_mu, da_sd))),
        })
df = pd.DataFrame(rows)
features = ['watch_hours_week', 'num_genres', 'days_active']
X_scaled = StandardScaler().fit_transform(df[features].values)

pca = PCA(random_state=42)
pca.fit(X_scaled)

print("PCA on CinemaStream user features (3 features):")
print(f"{'PC':>4}  {'Var explained':>14}  {'Cumulative':>10}")
cumul = 0.0
for i, v in enumerate(pca.explained_variance_ratio_, 1):
    cumul += v
    print(f"PC{i:>2}  {v:>10.3f} ({v:.1%})  {cumul:>9.1%}")

print("\nPC1 loadings — the 'engagement axis':")
for feat, load in zip(features, pca.components_[0]):
    print(f"  {feat:22s}: {load:+.3f}")

# Sample PC1 coordinates: low vs high engagement
coords = pca.transform(X_scaled)
print("\nSample PC1 score (engagement axis), first user from each plan tier:")
starts = [0, 168, 252]
for idx, plan in zip(starts, ['Free', 'Basic', 'Premium']):
    print(f"  {plan:8s}: PC1 = {coords[idx, 0]:+.2f}  PC2 = {coords[idx, 1]:+.2f}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
segments = [
    ('Free',    168, (5.0,  2.5), (2.0, 0.8), (8,  4)),
    ('Basic',    84, (12.0, 3.5), (4.0, 1.0), (15, 4)),
    ('Premium',  48, (22.0, 4.0), (6.0, 1.2), (22, 3)),
]
rows = []
for plan, n, (wh_mu, wh_sd), (ng_mu, ng_sd), (da_mu, da_sd) in segments:
    for _ in range(n):
        rows.append({
            'plan': plan,
            'watch_hours_week':  max(0.0, rng.normal(wh_mu, wh_sd)),
            'num_genres':        max(1,   round(rng.normal(ng_mu, ng_sd))),
            'days_active':       max(1,   round(rng.normal(da_mu, da_sd))),
        })

# Inject 3 bot-like accounts: 168 h/week = 24h/day for 7 days, single genre, active every day
for _ in range(3):
    rows.append({'plan': 'Free', 'watch_hours_week': 168.0,
                 'num_genres': 1, 'days_active': 30})

df = pd.DataFrame(rows)  # 303 rows total (300 + 3 bots)
features = ['watch_hours_week', 'num_genres', 'days_active']
X = StandardScaler().fit_transform(df[features].values)

# Fit on normal users only, predict on all 303
iso = IsolationForest(contamination=0.01, random_state=42)
iso.fit(X[:300])
preds = iso.predict(X)

n_flagged  = (preds == -1).sum()
bots_caught = (preds[300:] == -1).sum()
fp = (preds[:300] == -1).sum()
print(f"Total accounts:           {len(preds)}")
print(f"Flagged as anomalous:     {n_flagged}")
print(f"Bot accounts detected:    {bots_caught}/3")
print(f"False positives (normal): {fp}")

print("\nFlagged accounts:")
flagged_idx = np.where(preds == -1)[0]
for i in flagged_idx:
    r = df.iloc[i]
    print(f"  [{i:3d}] plan={r['plan']:8s} "
          f"watch={r['watch_hours_week']:6.1f}h/wk  "
          f"genres={r['num_genres']:2.0f}  "
          f"days={r['days_active']:2.0f}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 74b: Recommender Systems — Collaborative, Content-Based & Hybrid

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas scikit-learn

### 2.1 Popularity baseline and content-based filtering

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Small movie catalogue
movies = pd.DataFrame({
    'movie_id':    [101, 102, 103, 104, 105, 106, 107, 108],
    'title':       ['Singapura Dreaming', 'Rain Over Selat', "Tiger's Mouth",
                    'Metro Nights', 'The River Bends', 'Last Signal',
                    'Open Water', 'Monsoon Hearts'],
    'description': [
        'romantic drama singapore family dreams aspirations urban life',
        'drama rain monsoon singapore selat malay love story',
        'action thriller tiger crime singapore gangster chase',
        'crime thriller urban night singapore corruption chase gangster',
        'drama river slow family heritage rural tradition bends',
        'science fiction signal space isolation last frontier survival',
        'adventure survival ocean open water danger rescue thriller',
        'romantic drama monsoon hearts love loss malay culture rain',
    ],
    'n_views': [4500, 3200, 6100, 5800, 1200, 890, 2300, 2900],
})

# --- Popularity baseline ---
top_n = 3
popular = movies.nlargest(top_n, 'n_views')[['title', 'n_views']]
print("Popularity baseline — top 3:")
for _, r in popular.iterrows():
    print(f"  {r['title']:22s}  ({r['n_views']:,} views)")

# --- Content-based: TF-IDF cosine similarity ---
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['description'])
sim_matrix = cosine_similarity(tfidf_matrix)   # shape (8, 8)

def content_recommend(liked_movie_id, top_k=3):
    idx = movies.index[movies['movie_id'] == liked_movie_id].item()
    sims = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1]
    results = []
    for i in top_idx:
        if movies.iloc[i]['movie_id'] != liked_movie_id:
            results.append((movies.iloc[i]['title'], round(sims[i], 3)))
        if len(results) == top_k:
            break
    return results

print("\nContent-based: user liked 'Tiger\\'s Mouth' (action/crime) → recommend:")
for title, score in content_recommend(103):
    print(f"  {title:22s}  (similarity={score})")

print("\nContent-based: user liked 'Singapura Dreaming' (romance/drama) → recommend:")
for title, score in content_recommend(101):
    print(f"  {title:22s}  (similarity={score})")

### 2.2 User-based collaborative filtering

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Ratings matrix: rows=users, columns=movies 101-108
# 0 = not rated; 1-5 = rating
ratings = np.array([
    # 101  102  103  104  105  106  107  108
    [  5,   4,   0,   0,   3,   0,   0,   4],  # user 1 (Ravi — romance/family)
    [  0,   0,   5,   5,   0,   2,   4,   0],  # user 2 (action/thriller)
    [  4,   5,   0,   0,   4,   0,   0,   5],  # user 3 (romance/drama, like Ravi)
    [  0,   0,   4,   5,   0,   3,   5,   0],  # user 4 (like user 2)
    [  0,   3,   0,   0,   5,   0,   0,   3],  # user 5 (Siti — slow drama)
    [  3,   0,   0,   0,   0,   5,   0,   0],  # user 6 (Minh — eclectic/sci-fi)
], dtype=float)

movie_ids = [101, 102, 103, 104, 105, 106, 107, 108]
movie_titles = {
    101: 'Singapura Dreaming', 102: 'Rain Over Selat',
    103: "Tiger's Mouth",      104: 'Metro Nights',
    105: 'The River Bends',    106: 'Last Signal',
    107: 'Open Water',         108: 'Monsoon Hearts',
}

# Cosine similarity between users (treat 0-ratings as absent, not "rated zero")
user_sim = cosine_similarity(ratings)

def cf_recommend(user_idx, top_k=3):
    sims = user_sim[user_idx].copy()
    sims[user_idx] = -1   # exclude self
    most_similar = np.argsort(sims)[::-1][:2]  # top 2 similar users

    # Aggregate neighbour ratings for unwatched items
    unwatched = np.where(ratings[user_idx] == 0)[0]
    scores = {}
    for col in unwatched:
        neighbour_ratings = [ratings[nb, col] for nb in most_similar
                             if ratings[nb, col] > 0]
        if neighbour_ratings:
            scores[col] = (np.mean(neighbour_ratings), movie_ids[col])

    top = sorted(scores.items(), key=lambda x: x[1][0], reverse=True)[:top_k]
    return [(movie_titles[mid], round(avg, 2)) for _, (avg, mid) in top]

print("User-based CF for Ravi (user 1 — romance/family watcher):")
for title, avg in cf_recommend(0):
    print(f"  {title:22s}  (neighbour avg rating={avg})")

print("\nUser-based CF for Siti (user 5 — slow drama):")
for title, avg in cf_recommend(4):
    print(f"  {title:22s}  (neighbour avg rating={avg})")

# Cold-start: Minh has only rated 2 movies
print(f"\nMinh (user 6) has rated {(ratings[5]>0).sum()} movies.")
print("Neighbours found:", [i for i in np.argsort(user_sim[5])[::-1] if i != 5][:2])
print("→ Sparse history = poor CF signal; fall back to popularity baseline.")

### 2.3 Matrix factorization: SVD intuition

In [ ]:
import numpy as np

# Use the same 6×8 rating matrix from above
ratings = np.array([
    [5, 4, 0, 0, 3, 0, 0, 4],
    [0, 0, 5, 5, 0, 2, 4, 0],
    [4, 5, 0, 0, 4, 0, 0, 5],
    [0, 0, 4, 5, 0, 3, 5, 0],
    [0, 3, 0, 0, 5, 0, 0, 3],
    [3, 0, 0, 0, 0, 5, 0, 0],
], dtype=float)

# Truncated SVD with k=2 latent factors
U, sigma, Vt = np.linalg.svd(ratings, full_matrices=False)
k = 2
U_k     = U[:, :k]
sigma_k = np.diag(sigma[:k])
Vt_k    = Vt[:k, :]

# Reconstruct predicted ratings
R_hat = U_k @ sigma_k @ Vt_k
print("Predicted ratings (k=2 latent factors), rounded to 1 decimal:")
print("       " + "  ".join(f"M{i+1:02d}" for i in range(8)))
for i, row in enumerate(R_hat):
    print(f"  U{i+1}: " + "  ".join(f"{v:4.1f}" for v in row))

# Show latent factor for user 1 (Ravi) — the 2D taste vector
print(f"\nUser 1 (Ravi) latent taste vector (k=2): {U_k[0].round(3)}")
print(f"User 5 (Siti) latent taste vector (k=2): {U_k[4].round(3)}")
print(f"User 2 (action) latent taste vector:      {U_k[1].round(3)}")

# Cosine similarity between Ravi and Siti's latent vectors vs Ravi and action user
ravi_siti = np.dot(U_k[0], U_k[4]) / (np.linalg.norm(U_k[0]) * np.linalg.norm(U_k[4]))
ravi_action = np.dot(U_k[0], U_k[1]) / (np.linalg.norm(U_k[0]) * np.linalg.norm(U_k[1]))
print(f"\nLatent-space similarity:")
print(f"  Ravi ↔ Siti:         {ravi_siti:.3f}")
print(f"  Ravi ↔ action user:  {ravi_action:.3f}")

### 2.4 Precision@k and Recall@k

In [ ]:
def precision_at_k(recommended, relevant, k):
    top_k = set(recommended[:k])
    return len(top_k & relevant) / k

def recall_at_k(recommended, relevant, k):
    top_k = set(recommended[:k])
    return len(top_k & relevant) / len(relevant) if relevant else 0.0

# Ravi (user 1): relevant items = those he hasn't rated but neighbours rated highly
# Suppose ground truth: relevant = {105, 108} (the two he'd actually enjoy)
recommended = [108, 105, 102, 106, 107]  # system's ranked list
relevant = {105, 108}

print(f"Recommended: {recommended}")
print(f"Relevant:    {sorted(relevant)}")
print()
for k in [1, 2, 3, 5]:
    p = precision_at_k(recommended, relevant, k)
    r = recall_at_k(recommended, relevant, k)
    print(f"  @{k}: Precision={p:.2f}  Recall={r:.2f}")

## 3. CinemaStream in Practice

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Movie catalogue — canonical CinemaStream titles with descriptions
movies = pd.DataFrame({
    'movie_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'title': [
        'Singapura Dreaming', 'Rain Over Selat', "Tiger's Mouth",
        'Metro Nights', 'The River Bends', 'Last Signal',
        'Open Water', 'Monsoon Hearts',
    ],
    'description': [
        'romantic drama singapore family dreams aspirations urban',
        'drama monsoon singapore selat malay love loss romance',
        'action thriller crime singapore gangster chase pursuit',
        'crime thriller urban corruption night singapore chase',
        'drama heritage rural family river bends slow tradition',
        'science fiction survival space isolation signal distress',
        'adventure thriller ocean rescue survival open water danger',
        'romance malay monsoon hearts love culture drama loss rain',
    ],
    'n_views': [4500, 3200, 6100, 5800, 1200, 890, 2300, 2900],
})

# --- Content-based similarity matrix (TF-IDF on descriptions) ---
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['description'])
content_sim = cosine_similarity(tfidf_matrix)  # (8, 8)

# --- Synthetic cooccurrence matrix from watch_events ---
# Engagement cluster shapes (from Ch074a): Casual=Free, Engaged=Basic, Invested=Premium
# Casual users → mostly romance/drama (101, 102, 108)
# Engaged users → mix of genres (adds 103, 104, 105)
# Invested users → full catalogue including 105, 106, 107
rng = np.random.default_rng(42)

def hybrid_recommend(user_history, top_k=4, content_weight=0.5, collab_weight=0.5):
    """
    user_history: list of movie_ids the user has watched
    Returns top_k recommendations using a weighted hybrid.
    """
    watched_idx = [movies.index[movies['movie_id'] == mid].item()
                   for mid in user_history if mid in movies['movie_id'].values]
    if not watched_idx:
        return [(title, 0.0, 0.0) for title in movies.nlargest(top_k, 'n_views')['title'].tolist()]

    # Content score: avg similarity to all watched movies
    content_scores = content_sim[watched_idx].mean(axis=0)

    # Collaboration proxy: boost movies popular in same viewing-cluster
    # (simplified: use log-normalised global view counts as a stand-in)
    collab_scores = np.log1p(movies['n_views'].values)
    collab_scores = collab_scores / collab_scores.max()

    combined = content_weight * content_scores + collab_weight * collab_scores
    # Zero out watched movies
    for i in watched_idx:
        combined[i] = -1

    top_idx = np.argsort(combined)[::-1][:top_k]
    return [(movies.iloc[i]['title'],
             round(content_scores[i], 3),
             round(collab_scores[i], 3))
            for i in top_idx]

# --- Ravi (Premium, Casual→Invested, loves regional drama) ---
ravi_history = [101, 102]  # Singapura Dreaming + Rain Over Selat
print("Recommendations for Ravi (history: 101, 102 — regional romance):")
for title, c_score, p_score in hybrid_recommend(ravi_history):
    print(f"  {title:22s}  content={c_score:.3f}  popularity={p_score:.3f}")

# --- Siti (Basic, slow drama) ---
siti_history = [105, 102]  # The River Bends + Rain Over Selat
print("\nRecommendations for Siti (history: 102, 105 — slow/rural drama):")
for title, c_score, p_score in hybrid_recommend(siti_history):
    print(f"  {title:22s}  content={c_score:.3f}  popularity={p_score:.3f}")

# --- Minh (cold start: 0 history) ---
minh_history = []
print("\nMinh (cold start — no history): falls back to popularity baseline")
for title, c_score, p_score in hybrid_recommend(minh_history):
    print(f"  {title:22s}  (popularity only)")

In [ ]:
# Precision@k evaluation on a held-out "relevant" judgement
# Suppose the team has hand-labelled the ground truth for Ravi:
# After seeing his history, a curator says these are genuinely relevant: {108, 105}

def evaluate_hybrid(user_history, relevant_ids, k=4, cw=0.5):
    recs = hybrid_recommend(user_history, top_k=k, content_weight=cw, collab_weight=1-cw)
    rec_titles = [r[0] for r in recs] if recs and isinstance(recs[0], tuple) else recs
    relevant_titles = {movies.loc[movies['movie_id']==mid, 'title'].values[0]
                       for mid in relevant_ids}
    hits = sum(1 for t in rec_titles if t in relevant_titles)
    return hits / k, hits / len(relevant_ids)

print("Ravi held-out evaluation (relevant = Monsoon Hearts + The River Bends):")
print(f"{'weight':>8}  {'P@4':>6}  {'R@4':>6}")
for cw in [0.2, 0.5, 0.8]:
    p, r = evaluate_hybrid([101, 102], {108, 105}, k=4, cw=cw)
    print(f"  cw={cw:.1f}    {p:.2f}    {r:.2f}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 75: ML Experiment Tracking — MLflow, the Model Registry, Artifact Management, and Reproducibility

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    A[Training run] --> B[mlflow.start_run]
    B --> C[log_param\nmax_depth, seed, features]
    B --> D[log_metric\ncv_f1, cv_f1_std]
    B --> E[log_artifact\nmodel.joblib, plot]
    C --> F[(SQLite tracking store\ncinemastream-churn experiment)]
    D --> F
    E --> F
    F --> G[mlflow.search_runs\nreturns DataFrame]
    G --> H[Register best run\nModel Registry v1, v2...]
    H --> I[Alias @champion\npoints to current best]
    I --> J[models:/cinemastream-churn@champion\nstable deployment address]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install mlflow numpy scikit-learn

### 2.1 Your first tracked run

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow_demo.db")
mlflow.set_experiment("toy-experiment")

with mlflow.start_run(run_name="first-run") as run:
    mlflow.log_param("model_family", "decision_tree")
    mlflow.log_param("max_depth", 3)
    mlflow.log_metric("cv_accuracy", 0.877)
    mlflow.log_metric("cv_accuracy_std", 0.028)
    mlflow.set_tag("author", "employee_47")
    print(f"run_id: {run.info.run_id}")
print("Run logged to mlflow_demo.db")

```
run_id: bfac7b6053a44d5680cd622175f34b7f
Run logged to mlflow_demo.db
```

### 2.2 Logging the model itself — and getting it back

In [ ]:
import numpy as np
import mlflow.sklearn
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(n_samples=600, n_features=10, n_informative=4,
                           random_state=7)
tree = DecisionTreeClassifier(max_depth=3, random_state=7).fit(X, y)

with mlflow.start_run(run_name="tree-with-model") as run:
    mlflow.log_param("max_depth", 3)
    mlflow.log_metric("train_accuracy", tree.score(X, y))
    info = mlflow.sklearn.log_model(tree, name="model", input_example=X[:2])
    print(f"model_uri: {info.model_uri}")

loaded = mlflow.sklearn.load_model(info.model_uri)
print(f"Loaded model is identical: "
      f"{np.array_equal(tree.predict(X), loaded.predict(X))}")

```
model_uri: models:/m-be5a4ca83a334d26b4d25a9d422ca28f
Loaded model is identical: True
```

### 2.3 Your experiment history is a DataFrame

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
for depth in [2, 4, 8]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=7)
    scores = cross_val_score(model, X, y, cv=cv)
    with mlflow.start_run(run_name=f"tree-depth-{depth}"):
        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("cv_accuracy", scores.mean())
        mlflow.log_metric("cv_accuracy_std", scores.std())

runs = mlflow.search_runs(
    filter_string="metrics.cv_accuracy > 0.86",
    order_by=["metrics.cv_accuracy DESC"])
print(runs[["tags.mlflow.runName", "params.max_depth",
            "metrics.cv_accuracy"]].to_string(index=False))

```
tags.mlflow.runName params.max_depth  metrics.cv_accuracy
       tree-depth-8                8             0.878333
          first-run                3             0.877000
       tree-depth-4                4             0.875000
       tree-depth-2                2             0.861667
```

### 2.4 The model registry: names, versions, aliases

In [ ]:
result = mlflow.register_model(info.model_uri, "toy-classifier")
print(f"Registered: name={result.name}, version={result.version}")

client = mlflow.MlflowClient()
client.set_registered_model_alias("toy-classifier", "champion", result.version)

champion = mlflow.sklearn.load_model("models:/toy-classifier@champion")
print(f"Champion loaded, identical predictions: "
      f"{np.array_equal(tree.predict(X), champion.predict(X))}")

```
Successfully registered model 'toy-classifier'.
Created version '1' of model 'toy-classifier'.
Registered: name=toy-classifier, version=1
Champion loaded, identical predictions: True
```

## 3. CinemaStream in Practice

In [ ]:
import sys
sys.path.insert(0, ".")

import logging, os, warnings
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

from pathlib import Path
import mlflow
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score

from cinemastream.ml.churn.train import RANDOM_SEED
from cinemastream.ml.churn.hyperparameter_tuning import CV, load_split
from cinemastream.ml.churn.model_comparison import TUNED_RF_PARAMS

mlflow.set_tracking_uri("sqlite:///cinemastream/ml/mlflow.db")
artifact_location = Path("cinemastream/ml/mlruns").absolute().as_uri()
if mlflow.get_experiment_by_name("cinemastream-churn") is None:
    mlflow.create_experiment("cinemastream-churn",
                             artifact_location=artifact_location)
mlflow.set_experiment("cinemastream-churn")

X_train, X_test, y_train, y_test, names = load_split()

references = {
    "default-rf-ch068": (
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
        {"source_chapter": "068", "role": "historical-baseline"}),
    "tuned-rf-ch073": (
        RandomForestClassifier(**TUNED_RF_PARAMS),
        {"source_chapter": "073", "role": "champion"}),
    "gradient-boosting-ch074": (
        GradientBoostingClassifier(random_state=RANDOM_SEED),
        {"source_chapter": "074", "role": "challenger-untuned"}),
}
for run_name, (model, tags) in references.items():
    auc = cross_val_score(model, X_train, y_train, scoring="roc_auc", cv=CV)
    f1 = cross_val_score(model, X_train, y_train, scoring="f1", cv=CV)
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(model.get_params())
        mlflow.log_metric("cv_roc_auc", auc.mean())
        mlflow.log_metric("cv_roc_auc_std", auc.std())
        mlflow.log_metric("cv_f1", f1.mean())
        mlflow.log_metric("cv_f1_std", f1.std())
        mlflow.set_tags(tags)
    print(f"logged {run_name}: cv_roc_auc={auc.mean():.3f}, cv_f1={f1.mean():.3f}")

```
logged default-rf-ch068: cv_roc_auc=0.774, cv_f1=0.100
logged tuned-rf-ch073: cv_roc_auc=0.763, cv_f1=0.211
logged gradient-boosting-ch074: cv_roc_auc=0.768, cv_f1=0.067
```

In [ ]:
from cinemastream.ml.churn.hyperparameter_tuning import tune_optuna

study, cw_options = tune_optuna(X_train, y_train)
for trial in study.trials:
    with mlflow.start_run(run_name=f"optuna-trial-{trial.number:02d}"):
        mlflow.log_params(trial.params)
        mlflow.log_metric("cv_f1", trial.value)
        mlflow.set_tags({"search": "optuna-tpe-ch073",
                         "trial_number": str(trial.number)})
print(f"logged {len(study.trials)} Optuna trials "
      f"(best: trial {study.best_trial.number}, cv_f1={study.best_value:.3f})")

```
logged 40 Optuna trials (best: trial 12, cv_f1=0.211)
```

In [ ]:
runs = mlflow.search_runs(filter_string="tags.trial_number = '23'")
print(runs[["tags.mlflow.runName", "params.class_weight", "params.max_depth",
            "params.n_estimators", "params.min_samples_leaf",
            "metrics.cv_f1"]].to_string(index=False))

```
tags.mlflow.runName params.class_weight params.max_depth params.n_estimators params.min_samples_leaf  metrics.cv_f1
    optuna-trial-23                none               12                 364                       6            0.0
```

In [ ]:
import numpy as np
import mlflow.sklearn

champion = RandomForestClassifier(**TUNED_RF_PARAMS).fit(X_train, y_train)
with mlflow.start_run(run_name="champion-fit"):
    mlflow.log_params(TUNED_RF_PARAMS)
    mlflow.set_tag("decision_threshold", "0.3")
    info = mlflow.sklearn.log_model(champion, name="model",
                                    input_example=X_train[:2])
version = mlflow.register_model(info.model_uri, "cinemastream-churn").version

client = mlflow.MlflowClient()
client.set_registered_model_alias("cinemastream-churn", "champion", version)
client.set_model_version_tag("cinemastream-churn", version,
                             "decision_threshold", "0.3")
print(f"registered cinemastream-churn v{version} -> @champion")

challenger = GradientBoostingClassifier(random_state=RANDOM_SEED).fit(
    X_train, y_train)
with mlflow.start_run(run_name="challenger-fit"):
    mlflow.log_params(challenger.get_params())
    info_c = mlflow.sklearn.log_model(challenger, name="model",
                                      input_example=X_train[:2])
version_c = mlflow.register_model(info_c.model_uri, "cinemastream-churn").version
client.set_registered_model_alias("cinemastream-churn", "challenger", version_c)
print(f"registered cinemastream-churn v{version_c} -> @challenger")

loaded = mlflow.sklearn.load_model("models:/cinemastream-churn@champion")
same = np.array_equal(champion.predict_proba(X_test),
                      loaded.predict_proba(X_test))
print(f"champion reloaded from registry, identical predict_proba "
      f"on test features: {same}")

```
registered cinemastream-churn v1 -> @champion
registered cinemastream-churn v2 -> @challenger
champion reloaded from registry, identical predict_proba on test features: True
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 75a: Data & Model Versioning — DVC, Artifacts & Reproducibility

## 0. Where You Are

## 1. The Concept

```
dvc init                   # initialise DVC in the Git repo (creates .dvc/)
dvc add data/myfile.csv    # version a file (creates myfile.csv.dvc, updates .dvc/cache/)
dvc push                   # push cached data to remote storage
dvc pull                   # restore data files from remote by their cached hashes
```

```yaml
# churn_training.csv.dvc
outs:
- md5: a3f4c2e1b9d7...
  size: 148234
  path: churn_training.csv
```

## 2. Theory & Mechanics

### 2.1 The problem: undetected file mutation

In [ ]:
import hashlib
import tempfile
import os
from pathlib import Path

def file_md5(filepath: str) -> str:
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

# Simulate a training CSV being silently overwritten
with tempfile.TemporaryDirectory() as tmpdir:
    data_path = Path(tmpdir) / 'churn_training.csv'

    # Version 1: original training data (200 rows)
    v1_content = "user_id,tenure,plan,watch_hours,churned\n"
    v1_content += "\n".join(
        f"{i},{'%.1f' % (i * 0.5)},{'Free' if i%3==0 else 'Basic'},{i%20},{i%7==0}"
        for i in range(1, 201)
    )
    data_path.write_text(v1_content)
    hash_v1 = file_md5(str(data_path))
    size_v1 = data_path.stat().st_size
    print(f"Version 1 — MD5: {hash_v1}  size: {size_v1:,} bytes  rows: 200")

    # Simulate the MLflow run locking this hash
    locked_hash = hash_v1
    print(f"Champion model locked to hash: {locked_hash}")

    # Someone overwrites the file with 220 rows (includes leakage from test period)
    v2_content = "user_id,tenure,plan,watch_hours,churned\n"
    v2_content += "\n".join(
        f"{i},{'%.1f' % (i * 0.5)},{'Free' if i%3==0 else 'Basic'},{i%20},{i%7==0}"
        for i in range(1, 221)  # 20 extra rows from "the future"
    )
    data_path.write_text(v2_content)
    hash_v2 = file_md5(str(data_path))
    print(f"\nVersion 2 — MD5: {hash_v2}  rows: 220")
    print(f"Hashes match: {hash_v1 == hash_v2}")
    print()
    if hash_v1 != hash_v2:
        print("⚠  DATA CONTAMINATION DETECTED")
        print("   The file was modified after the champion was trained.")
        print("   Any retrain on this file will see different data.")
        print(f"   Expected: {locked_hash}")
        print(f"   Actual:   {hash_v2}")

### 2.2 Content-addressed storage: DVC's cache mechanism

In [ ]:
import hashlib
import shutil
import json
from pathlib import Path
import tempfile

class MinimalDVCCache:
    """Minimal simulation of DVC's content-addressed cache."""

    def __init__(self, cache_dir: str):
        self.cache = Path(cache_dir)
        self.cache.mkdir(parents=True, exist_ok=True)

    def _md5(self, filepath: Path) -> str:
        h = hashlib.md5()
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                h.update(chunk)
        return h.hexdigest()

    def add(self, filepath: str) -> dict:
        src = Path(filepath)
        md5 = self._md5(src)
        cache_path = self.cache / md5[:2] / md5[2:]
        cache_path.parent.mkdir(exist_ok=True)
        shutil.copy2(src, cache_path)
        dvc_entry = {'md5': md5, 'size': src.stat().st_size, 'path': src.name}
        print(f"Cached: {src.name}  →  cache/{md5[:2]}/{md5[2:]}  ({src.stat().st_size:,} bytes)")
        return dvc_entry

    def checkout(self, dvc_entry: dict, dest_dir: str):
        md5 = dvc_entry['md5']
        cache_path = self.cache / md5[:2] / md5[2:]
        dest = Path(dest_dir) / dvc_entry['path']
        shutil.copy2(cache_path, dest)
        print(f"Restored: {dvc_entry['path']}  from cache/{md5[:2]}/{md5[2:]}")
        return dest

with tempfile.TemporaryDirectory() as tmpdir:
    cache_dir = Path(tmpdir) / '.dvc_cache'
    work_dir  = Path(tmpdir) / 'work'
    work_dir.mkdir()

    cache = MinimalDVCCache(str(cache_dir))

    # Write two versions of the training CSV
    v1_path = work_dir / 'churn_training.csv'
    v1_path.write_text("user_id,churned\n" + "\n".join(f"{i},{i%7==0}" for i in range(200)))
    entry_v1 = cache.add(str(v1_path))
    print(f".dvc pointer: {json.dumps(entry_v1, indent=2)}")

    # Simulate update
    v1_path.write_text("user_id,churned\n" + "\n".join(f"{i},{i%7==0}" for i in range(220)))
    entry_v2 = cache.add(str(v1_path))
    print(f"\nUpdated .dvc pointer: md5={entry_v2['md5'][:12]}...")

    # Restore v1 from cache
    restore_dir = Path(tmpdir) / 'restored'
    restore_dir.mkdir()
    cache.checkout(entry_v1, str(restore_dir))
    restored_lines = (restore_dir / 'churn_training.csv').read_text().count('\n')
    print(f"Restored file has {restored_lines} lines (original 200 data rows + header)")

### 2.3 Version manifest: linking data hash to model run

In [ ]:
import json
import hashlib
import datetime
from pathlib import Path
import tempfile

def compute_md5(content: str) -> str:
    return hashlib.md5(content.encode()).hexdigest()

# Simulate a version manifest that would be written at training time
def create_version_manifest(
    git_commit: str,
    data_hash: str,
    data_path: str,
    mlflow_run_id: str,
    metrics: dict,
    notes: str = ''
) -> dict:
    return {
        'created_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'git_commit': git_commit,
        'data': {
            'path': data_path,
            'dvc_md5': data_hash,
            'reproduce_cmd': f'dvc checkout {data_path}.dvc',
        },
        'experiment': {
            'mlflow_run_id': mlflow_run_id,
            'metrics': metrics,
        },
        'notes': notes,
    }

# Canonical champion manifest for CinemaStream churn model
training_csv_content = "user_id,churned\n" + "\n".join(
    f"{i},{i%7==0}" for i in range(200)
)
data_hash = compute_md5(training_csv_content)

manifest = create_version_manifest(
    git_commit   = 'e764c35',          # from session git log
    data_hash    = data_hash,
    data_path    = 'cinemastream/data/churn_training.csv',
    mlflow_run_id= 'run-ch073-optuna-best',
    metrics      = {'cv_f1': 0.211, 'test_auc': 0.829,
                    'test_recall': 0.600, 'threshold': 0.3},
    notes        = 'Champion promoted 2026-06-16. Trained on 200-user pilot export.'
)

print(json.dumps(manifest, indent=2))
print(f"\nTo reproduce: git checkout {manifest['git_commit']} && "
      f"{manifest['data']['reproduce_cmd']} && python cinemastream/ml/churn/train.py")

## 3. CinemaStream in Practice

In [ ]:
import hashlib
import json
from pathlib import Path
import tempfile

# Simulate the CinemaStream scenario:
# Champion was trained on 200-row export (Q3 data only)
# Someone refreshed with 220-row export (Q3 + 2 weeks of Q4 = includes test-period labels)

def md5_of_string(s: str) -> str:
    return hashlib.md5(s.encode()).hexdigest()

# The original export
original_csv = "user_id,plan,churned\n" + "\n".join(
    f"{i},{'Free' if i % 3 == 0 else 'Basic' if i % 3 == 1 else 'Premium'},{int(i % 7 == 0)}"
    for i in range(1, 201)
)

# The "refreshed" export (20 extra rows that contain Q4 churn outcomes)
refreshed_csv = "user_id,plan,churned\n" + "\n".join(
    f"{i},{'Free' if i % 3 == 0 else 'Basic' if i % 3 == 1 else 'Premium'},{int(i % 7 == 0)}"
    for i in range(1, 221)
)

hash_original  = md5_of_string(original_csv)
hash_refreshed = md5_of_string(refreshed_csv)

# Manifest written at champion promotion time
champion_manifest = {
    'data_dvc_md5': hash_original,
    'mlflow_run_id': 'run-ch073-optuna-best',
    'metrics': {'cv_f1': 0.211, 'test_auc': 0.829},
}

# What's on disk now
current_hash = hash_refreshed

print("Champion manifest (committed to Git at promotion):")
print(f"  data_dvc_md5 : {champion_manifest['data_dvc_md5'][:16]}...")
print(f"  cv_f1        : {champion_manifest['metrics']['cv_f1']}")
print(f"  test_auc     : {champion_manifest['metrics']['test_auc']}")
print()
print(f"Current file hash : {current_hash[:16]}...")
print(f"Hashes match      : {current_hash == champion_manifest['data_dvc_md5']}")
print()
if current_hash != champion_manifest['data_dvc_md5']:
    print("ROOT CAUSE IDENTIFIED:")
    print("  churn_training.csv was overwritten with a 220-row export.")
    print("  20 extra rows include Q4 churn labels (test-period data).")
    print("  Training on this file leaks future labels into cross-validation.")
    print("  cv_f1 dropped 0.211 → 0.183 because model can no longer")
    print("  'see' the Q4 patterns it was accidentally trained to memorise.")
    print()
    print("FIX:")
    print("  dvc checkout cinemastream/data/churn_training.csv.dvc")
    print("  (restores the exact 200-row Q3 export from DVC cache)")

In [ ]:
import hashlib

def gate_champion_promotion(
    candidate_manifest: dict,
    current_data_path: str
) -> tuple[bool, str]:
    """
    Returns (approved, reason).
    Blocks promotion if the data on disk doesn't match the manifest's DVC hash.
    """
    with open(current_data_path, 'rb') as f:
        h = hashlib.md5()
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
        current_hash = h.hexdigest()

    locked_hash = candidate_manifest.get('data_dvc_md5', '')
    if current_hash != locked_hash:
        return False, (
            f"DATA HASH MISMATCH — promotion blocked.\n"
            f"  Manifest locked to: {locked_hash[:16]}...\n"
            f"  Current file hash:  {current_hash[:16]}...\n"
            f"  Run `dvc checkout` to restore the training data before promoting."
        )

    metrics = candidate_manifest.get('metrics', {})
    if metrics.get('test_auc', 0) < 0.75:
        return False, f"AUC {metrics['test_auc']} below promotion threshold (0.75)."

    return True, (
        f"Promotion approved.\n"
        f"  Git commit:    {candidate_manifest.get('git_commit', 'unknown')}\n"
        f"  Data hash:     {locked_hash[:16]}... ✓\n"
        f"  test_auc:      {metrics.get('test_auc')} ✓\n"
        f"  MLflow run:    {candidate_manifest.get('mlflow_run_id')}"
    )

import tempfile, os

with tempfile.NamedTemporaryFile(mode='wb', suffix='.csv', delete=False) as f:
    content = "user_id,plan,churned\n" + "\n".join(
        f"{i},Basic,{int(i % 7 == 0)}" for i in range(1, 201)
    )
    f.write(content.encode())
    tmp_path = f.name

locked_hash = hashlib.md5(content.encode()).hexdigest()
manifest = {
    'git_commit': 'e764c35',
    'data_dvc_md5': locked_hash,
    'mlflow_run_id': 'run-ch073-optuna-best',
    'metrics': {'cv_f1': 0.211, 'test_auc': 0.829},
}

approved, reason = gate_champion_promotion(manifest, tmp_path)
print(f"Gate result: {'APPROVED' if approved else 'BLOCKED'}")
print(reason)
os.unlink(tmp_path)

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 76: Model Serialization — pickle, joblib, and ONNX

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy scikit-learn

### 2.1 pickle: convenient, full-fidelity, in-process

In [ ]:
import pickle
import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(n_samples=600, n_features=10, n_informative=4,
                           random_state=7)
tree = DecisionTreeClassifier(max_depth=3, random_state=7).fit(X, y)

blob = pickle.dumps(tree)
print(f"type: {type(blob).__name__}, size: {len(blob):,} bytes")

restored = pickle.loads(blob)
print(f"restored: {restored}")
print(f"identical predictions: "
      f"{np.array_equal(tree.predict(X), restored.predict(X))}")

```
type: bytes, size: 2,308 bytes
restored: DecisionTreeClassifier(max_depth=3, random_state=7)
identical predictions: True
```

### 2.2 The danger, demonstrated

In [ ]:
class TotallyAModel:
    def __reduce__(self):
        return (print, ("!! arbitrary code just ran while 'loading a model' !!",))

blob = pickle.dumps(TotallyAModel())
print(f"file received: {len(blob)} bytes, looks like any other .pkl")
print("loading it...")
obj = pickle.loads(blob)
print(f"loads() returned: {obj!r}")

```
file received: 93 bytes, looks like any other .pkl
loading it...
!! arbitrary code just ran while 'loading a model' !!
loads() returned: None
```

### 2.3 joblib: faster and smaller for array-heavy models

In [ ]:
import os, time, joblib
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=7).fit(X, y)

t0 = time.perf_counter()
with open("rf.pkl", "wb") as f:
    pickle.dump(rf, f)
t1 = time.perf_counter()
joblib.dump(rf, "rf.joblib")
t2 = time.perf_counter()
joblib.dump(rf, "rf_compressed.joblib", compress=3)
t3 = time.perf_counter()

for path, secs in [("rf.pkl", t1 - t0), ("rf.joblib", t2 - t1),
                   ("rf_compressed.joblib", t3 - t2)]:
    print(f"{path:22s} {os.path.getsize(path) / 1024:>9.1f} KB   "
          f"saved in {secs * 1000:>6.1f} ms")

reloaded = joblib.load("rf_compressed.joblib")
print(f"compressed joblib round trip identical: "
      f"{np.array_equal(rf.predict(X), reloaded.predict(X))}")

```
rf.pkl                   2303.5 KB   saved in    5.4 ms
rf.joblib                2326.7 KB   saved in   35.3 ms
rf_compressed.joblib       483.8 KB   saved in   47.8 ms
compressed joblib round trip identical: True
```

### 2.4 ONNX: a portable graph that runs without sklearn

In [ ]:
from skl2onnx import to_onnx
import onnxruntime as ort

onx = to_onnx(rf, X[:1].astype(np.float32),
              options={id(rf): {"zipmap": False}})
onnx_bytes = onx.SerializeToString()
print(f"ONNX graph size: {len(onnx_bytes) / 1024:.1f} KB")

sess = ort.InferenceSession(onnx_bytes, providers=["CPUExecutionProvider"])
print(f"runtime inputs:  {[i.name for i in sess.get_inputs()]}")
print(f"runtime outputs: {[o.name for o in sess.get_outputs()]}")

labels, probas = sess.run(None, {"X": X.astype(np.float32)})
print(f"labels identical to sklearn:    "
      f"{np.array_equal(labels, rf.predict(X))}")
print(f"probabilities match (atol=1e-4): "
      f"{np.allclose(probas, rf.predict_proba(X), atol=1e-4)}")
print(f"max probability difference:     "
      f"{np.abs(probas - rf.predict_proba(X)).max():.2e}")

```
ONNX graph size: 1034.9 KB
runtime inputs:  ['X']
runtime outputs: ['label', 'probabilities']
labels identical to sklearn:    True
probabilities match (atol=1e-4): True
max probability difference:     6.56e-07
```

## 3. CinemaStream in Practice

In [ ]:
import sys
sys.path.insert(0, ".")
import warnings; warnings.filterwarnings("ignore")

import os, pickle, time
from pathlib import Path
import joblib, numpy as np
from sklearn.ensemble import RandomForestClassifier

from cinemastream.ml.churn.train import RANDOM_SEED
from cinemastream.ml.churn.hyperparameter_tuning import load_split
from cinemastream.ml.churn.model_comparison import TUNED_RF_PARAMS

ARTIFACT_DIR = Path("cinemastream/ml/churn/artifacts")
DECISION_THRESHOLD = 0.3

X_train, X_test, y_train, y_test, names = load_split()
model = RandomForestClassifier(**TUNED_RF_PARAMS).fit(X_train, y_train)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

paths = {}
for label, fn in [
    ("pickle", lambda p: pickle.dump(model, open(p, "wb"))),
    ("joblib", lambda p: joblib.dump(model, p)),
    ("joblib (compress=3)", lambda p: joblib.dump(model, p, compress=3)),
]:
    suffix = "pkl" if label == "pickle" else "joblib"
    name = "champion_compressed.joblib" if "compress" in label else f"champion.{suffix}"
    path = ARTIFACT_DIR / name
    t0 = time.perf_counter(); fn(path); secs = time.perf_counter() - t0
    paths[label] = path
    print(f"{label:22s} {os.path.getsize(path)/1024:>7.1f} KB {secs*1000:>7.1f} ms")

```
pickle                   581.7 KB     6.1 ms
joblib                   612.8 KB    44.4 ms
joblib (compress=3)      147.5 KB    46.1 ms
```

In [ ]:
for label, path in paths.items():
    loaded = joblib.load(path) if "joblib" in label else pickle.load(open(path, "rb"))
    same = np.array_equal(model.predict_proba(X_test), loaded.predict_proba(X_test))
    print(f"{label:22s} reload identical predict_proba: {same}")

```
pickle                 reload identical predict_proba: True
joblib                 reload identical predict_proba: True
joblib (compress=3)    reload identical predict_proba: True
```

In [ ]:
from skl2onnx import to_onnx
import onnxruntime as ort

onx = to_onnx(model, X_test[:1].astype(np.float32),
              options={id(model): {"zipmap": False}})
onnx_path = ARTIFACT_DIR / "champion.onnx"
onnx_path.write_bytes(onx.SerializeToString())
print(f"ONNX graph written: {os.path.getsize(onnx_path)/1024:.1f} KB")

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
labels, probas = sess.run(None, {input_name: X_test.astype(np.float32)})

sk_proba = model.predict_proba(X_test)
print(f"ONNX labels match sklearn:           {np.array_equal(labels, model.predict(X_test))}")
print(f"ONNX probabilities match (atol=1e-4): {np.allclose(probas, sk_proba, atol=1e-4)}")
print(f"max probability difference:           {np.abs(probas - sk_proba).max():.2e}")

sk_flags = (sk_proba[:, 1] >= DECISION_THRESHOLD).astype(int)
onnx_flags = (probas[:, 1] >= DECISION_THRESHOLD).astype(int)
print(f"churn flags @ {DECISION_THRESHOLD} identical (sklearn vs ONNX): "
      f"{np.array_equal(sk_flags, onnx_flags)}")

```
ONNX graph written: 220.0 KB
ONNX labels match sklearn:           True
ONNX probabilities match (atol=1e-4): True
max probability difference:           2.16e-07
churn flags @ 0.3 identical (sklearn vs ONNX): True
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 77: ML Pipeline Integration — REST APIs with FastAPI, and Putting the Churn Model in Production

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    A[POST /predict\nJSON request] --> B[Pydantic validation\n422 on bad input]
    B --> C[Feature engineering\nfrozen training constants]
    C --> D[Load @champion\nfrom Model Registry]
    D --> E[model.predict_proba]
    E --> F[Apply threshold\nchurn yes/no]
    F --> G[Structured log\nevery prediction]
    F --> H[200 JSON response]
    G --> I[Health probe /health]
    G --> J[Drift monitor\nprediction stream]
    J --> K{Score drifting?}
    K -->|Yes| L[Alert on-call]
    K -->|No| M[Healthy]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install fastapi pydantic

### 2.1 A minimal API

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/ping")
def ping():
    return {"message": "pong"}

@app.get("/area")
def area(width: float, height: float):
    return {"area": width * height}

client = TestClient(app)
print("GET /ping        ->", client.get("/ping").json())
print("GET /area?w=3&h=4 ->", client.get("/area", params={"width": 3, "height": 4}).json())

```
GET /ping        -> {'message': 'pong'}
GET /area?w=3&h=4 -> {'area': 12.0}
```

### 2.2 Typed request bodies and automatic validation

In [ ]:
from pydantic import BaseModel, Field

class Rectangle(BaseModel):
    width: float = Field(gt=0, le=100)
    height: float = Field(gt=0, le=100)
    label: str = "unnamed"

app2 = FastAPI()

@app2.post("/rectangle")
def make_rectangle(rect: Rectangle):
    return {"label": rect.label, "area": rect.width * rect.height}

client2 = TestClient(app2)
ok = client2.post("/rectangle", json={"width": 5, "height": 6, "label": "A"})
print("valid request   -> HTTP", ok.status_code, ok.json())

bad = client2.post("/rectangle", json={"width": -2, "height": 6})
print("width=-2        -> HTTP", bad.status_code,
      "->", bad.json()["detail"][0]["type"], "on", bad.json()["detail"][0]["loc"])

missing = client2.post("/rectangle", json={"width": 5})
print("height missing  -> HTTP", missing.status_code,
      "->", missing.json()["detail"][0]["type"], "on", missing.json()["detail"][0]["loc"])

wrong_type = client2.post("/rectangle", json={"width": "wide", "height": 6})
print("width='wide'    -> HTTP", wrong_type.status_code,
      "->", wrong_type.json()["detail"][0]["type"])

```
valid request   -> HTTP 200 {'label': 'A', 'area': 30.0}
width=-2        -> HTTP 422 -> greater_than on ['body', 'width']
height missing  -> HTTP 422 -> missing on ['body', 'height']
width='wide'    -> HTTP 422 -> float_parsing
```

### 2.3 Response models, status codes, and health checks

In [ ]:
from typing import Literal

class AreaResponse(BaseModel):
    label: str
    area: float
    unit: Literal["sq_units"] = "sq_units"

app3 = FastAPI()

@app3.get("/health")
def health():
    return {"status": "ok"}

@app3.post("/rectangle", response_model=AreaResponse)
def make_rectangle3(rect: Rectangle):
    return {"label": rect.label, "area": rect.width * rect.height,
            "internal_debug": "this is dropped by the response_model"}

client3 = TestClient(app3)
print("GET /health     ->", client3.get("/health").json())
resp = client3.post("/rectangle", json={"width": 2, "height": 3, "label": "B"})
print("POST /rectangle -> response_model filters output:", resp.json())

```
GET /health     -> {'status': 'ok'}
POST /rectangle -> response_model filters output: {'label': 'B', 'area': 6.0, 'unit': 'sq_units'}
```

### 2.4 Structured logging: every request, replayable

In [ ]:
import json, logging, sys, time, uuid

logger = logging.getLogger("toy-api")
logger.setLevel(logging.INFO)
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(h)

app4 = FastAPI()

@app4.post("/rectangle")
def make_rectangle4(rect: Rectangle):
    request_id = str(uuid.uuid4())
    t0 = time.perf_counter()
    result = rect.width * rect.height
    latency_ms = round((time.perf_counter() - t0) * 1000, 3)
    logger.info(json.dumps({"request_id": request_id[:8], "label": rect.label,
                            "area": result, "latency_ms": latency_ms}))
    return {"area": result}

client4 = TestClient(app4)
client4.post("/rectangle", json={"width": 4, "height": 4, "label": "C"})

```
{"request_id": "01a443ff", "label": "C", "area": 16.0, "latency_ms": 0.001}
```

## 3. CinemaStream in Practice

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class ChurnRequest(BaseModel):
    country: Literal["SG", "MY", "ID", "PH", "TH", "VN", "IN"]
    plan: Literal["Free", "Basic", "Premium"]
    tenure_months: int = Field(ge=0, le=120)
    watch_minutes_avg: float = Field(ge=0, le=1000)
    days_since_last_watch: int = Field(ge=0, le=365)
    support_tickets_count: int = Field(ge=0, le=50)
    monthly_spend_sgd: float | None = Field(default=None, ge=0, le=200)

class ChurnResponse(BaseModel):
    request_id: str
    model_version: str
    churn_probability: float
    will_churn: bool
    threshold: float

In [ ]:
import numpy as np

SPEND_CAP = 38.1925                          # Ch070 IQR upper bound -- FROZEN
SPEND_MEDIAN_BY_PLAN = {"Free": 0.0050, "Basic": 12.9100, "Premium": 19.9450}

def engineer_serving_features(req):
    engagement = round(req.watch_minutes_avg / (req.days_since_last_watch + 1), 3)
    established = 1.0 if 12 <= req.tenure_months < 24 else 0.0
    spend = (req.monthly_spend_sgd if req.monthly_spend_sgd is not None
             else SPEND_MEDIAN_BY_PLAN[req.plan])       # impute with FROZEN median
    spend = min(spend, SPEND_CAP)                       # cap at FROZEN bound
    return np.array([[req.watch_minutes_avg, float(req.days_since_last_watch),
                      float(req.tenure_months), float(req.support_tickets_count),
                      engagement, 1.0 if req.country == "TH" else 0.0,
                      1.0 if req.country == "VN" else 0.0, established, spend]],
                    dtype=object)

In [ ]:
import joblib

DECISION_THRESHOLD = 0.3                       # Ch073's cutoff -- config
MODEL_VERSION = "cinemastream-churn@champion"  # Ch075 registry alias
MODEL = joblib.load("cinemastream/ml/churn/artifacts/champion_compressed.joblib")

In [ ]:
from collections import deque
from fastapi import FastAPI

app = FastAPI(title="CinemaStream Churn Scoring", version="1.0.0")
RECENT = deque(maxlen=500)   # last 500 predictions, for the /metrics window

@app.get("/health")
def health():
    # liveness, not correctness (Ch052): a 200 means "can serve"
    return {"status": "ok", "model_version": MODEL_VERSION, "threshold": DECISION_THRESHOLD}

@app.get("/metrics")
def metrics():
    if not RECENT:
        return {"predictions": 0}
    probs = [r["churn_probability"] for r in RECENT]
    flagged = sum(r["will_churn"] for r in RECENT)
    return {
        "predictions": len(RECENT),
        "flagged_rate": round(flagged / len(RECENT), 4),
        "mean_churn_probability": round(float(np.mean(probs)), 4),
        "p95_latency_ms": round(float(np.percentile([r["latency_ms"] for r in RECENT], 95)), 2),
        "model_version": MODEL_VERSION,
    }

print("Service ready:", app.title)

```
Service ready: CinemaStream Churn Scoring
```

In [ ]:
@app.post("/predict", response_model=ChurnResponse)
def predict(req: ChurnRequest):
    request_id = str(uuid.uuid4())
    t0 = time.perf_counter()
    features = engineer_serving_features(req)
    proba = float(MODEL.predict_proba(features)[0, 1])
    will_churn = proba >= DECISION_THRESHOLD
    latency_ms = round((time.perf_counter() - t0) * 1000, 2)
    logger.info(json.dumps({"request_id": request_id, "model_version": MODEL_VERSION,
        "country": req.country, "plan": req.plan, "churn_probability": round(proba, 4),
        "will_churn": will_churn, "latency_ms": latency_ms}))
    RECENT.append({"churn_probability": round(proba, 4), "will_churn": will_churn,
                   "latency_ms": latency_ms})
    return ChurnResponse(request_id=request_id, model_version=MODEL_VERSION,
        churn_probability=round(proba, 4), will_churn=will_churn,
        threshold=DECISION_THRESHOLD)

In [ ]:
from fastapi.testclient import TestClient
client = TestClient(app)

print("GET /health ->", client.get("/health").json())

loyal = {"country": "SG", "plan": "Basic", "tenure_months": 32,
         "watch_minutes_avg": 79.4, "days_since_last_watch": 36,
         "support_tickets_count": 2, "monthly_spend_sgd": 12.72}
r1 = client.post("/predict", json=loyal).json()
print(f"loyal user:   prob={r1['churn_probability']}, will_churn={r1['will_churn']}")

atrisk = {"country": "TH", "plan": "Basic", "tenure_months": 4,
          "watch_minutes_avg": 91.1, "days_since_last_watch": 55,
          "support_tickets_count": 0, "monthly_spend_sgd": 12.3}
r2 = client.post("/predict", json=atrisk).json()
print(f"at-risk user: prob={r2['churn_probability']}, will_churn={r2['will_churn']}")

no_spend = dict(atrisk); no_spend.pop("monthly_spend_sgd")
r3 = client.post("/predict", json=no_spend).json()
print(f"spend omitted -> imputed: prob={r3['churn_probability']}")

bad = dict(loyal); bad["tenure_months"] = -5
print(f"tenure=-5:    HTTP {client.post('/predict', json=bad).status_code}")
bad2 = dict(loyal); bad2["country"] = "US"
print(f"country='US': HTTP {client.post('/predict', json=bad2).status_code}")

print("GET /metrics ->", client.get("/metrics").json())

```
GET /health -> {'status': 'ok', 'model_version': 'cinemastream-churn@champion', 'threshold': 0.3}
loyal user:   prob=0.0119, will_churn=False
at-risk user: prob=0.6737, will_churn=True
spend omitted -> imputed: prob=0.6199
tenure=-5:    HTTP 422
country='US': HTTP 422
GET /metrics -> {'predictions': 3, 'flagged_rate': 0.6667, 'mean_churn_probability': 0.4352, 'p95_latency_ms': 5.44, 'model_version': 'cinemastream-churn@champion'}
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
@app.post("/predict")
def predict(payload: dict):
    model = joblib.load("model.pkl")
    features = [payload["a"], payload["b"], payload["c"]]
    return {"score": model.predict_proba([features])[0][1]}

---

# Chapter 77a: Feature Stores — Offline/Online Features & Training-Serving Skew

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas scikit-learn

### 2.1 Training-serving skew: simulating the failure

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
n = 200

# Raw feature: spend_sgd (skewed, some outliers)
spend_raw = np.abs(rng.normal(15, 8, n))
spend_raw[rng.integers(0, n, 5)] = 80  # 5 outlier accounts

# --- TRAINING pipeline: clips spend at 95th percentile ---
cap_95 = np.percentile(spend_raw, 95)
spend_train = np.clip(spend_raw, 0, cap_95)

# Simulated labels (churned if low spend)
y = (spend_raw < 10).astype(int)
X_train = spend_train.reshape(-1, 1)

model = LogisticRegression(random_state=42)
model.fit(X_train, y)

# Evaluate on "test set" using TRAINING feature logic (correct)
test_spend_raw = np.abs(rng.normal(15, 8, 40))
test_spend_raw[0] = 85   # one high-spend outlier in test

spend_test_correct = np.clip(test_spend_raw, 0, cap_95)
spend_test_skewed  = test_spend_raw  # SERVING bug: forgot to clip

preds_correct = model.predict(spend_test_correct.reshape(-1, 1))
preds_skewed  = model.predict(spend_test_skewed.reshape(-1, 1))

disagreements = (preds_correct != preds_skewed).sum()
print(f"Training cap (95th percentile): S${cap_95:.2f}")
print(f"Test predictions that DISAGREE due to skew: {disagreements}/{len(preds_correct)}")
print(f"  High-spend user (raw={test_spend_raw[0]:.1f}):")
print(f"    Correct (clipped to {min(test_spend_raw[0], cap_95):.2f})  → {preds_correct[0]}")
print(f"    Skewed  (raw {test_spend_raw[0]:.1f} passed through) → {preds_skewed[0]}")
print()
print("The model was never trained on spend > cap_95.")
print("Passing raw spend silently changes predictions for high-spend users.")

### 2.2 Point-in-time correct feature retrieval

In [ ]:
import pandas as pd
from datetime import datetime, timezone

# Subscription history: plan changes over time
subscription_history = pd.DataFrame([
    {'user_id': 1, 'plan': 'Free',    'valid_from': '2025-01-01', 'valid_to': '2025-06-01'},
    {'user_id': 1, 'plan': 'Basic',   'valid_from': '2025-06-01', 'valid_to': '2025-10-01'},
    {'user_id': 1, 'plan': 'Premium', 'valid_from': '2025-10-01', 'valid_to': None},
    {'user_id': 2, 'plan': 'Free',    'valid_from': '2025-01-01', 'valid_to': None},
])
subscription_history['valid_from'] = pd.to_datetime(subscription_history['valid_from'])
subscription_history['valid_to']   = pd.to_datetime(
    subscription_history['valid_to'].fillna('2099-01-01')
)

# Training events: churn signals recorded at specific timestamps
training_events = pd.DataFrame([
    {'user_id': 1, 'event_ts': '2025-04-15', 'churned': 0},  # Free at this time
    {'user_id': 1, 'event_ts': '2025-08-20', 'churned': 0},  # Basic at this time
    {'user_id': 1, 'event_ts': '2025-11-10', 'churned': 1},  # Premium at this time
    {'user_id': 2, 'event_ts': '2025-09-01', 'churned': 1},  # Free at this time
])
training_events['event_ts'] = pd.to_datetime(training_events['event_ts'])

def get_plan_at_time(user_id: int, ts: pd.Timestamp, history: pd.DataFrame) -> str:
    """Point-in-time correct plan lookup."""
    rows = history[
        (history['user_id'] == user_id) &
        (history['valid_from'] <= ts) &
        (history['valid_to'] > ts)
    ]
    return rows['plan'].iloc[0] if len(rows) else 'Unknown'

# Current plan (naive — what SERVING would return if it just read a user table)
current_plan_lookup = {1: 'Premium', 2: 'Free'}

print("Point-in-time vs naive feature retrieval:")
print(f"{'user':>5}  {'event_ts':>12}  {'pit_correct':>12}  {'naive_current':>14}  {'match':>5}")
for _, row in training_events.iterrows():
    pit   = get_plan_at_time(row['user_id'], row['event_ts'], subscription_history)
    naive = current_plan_lookup[row['user_id']]
    match = pit == naive
    print(f"  {row['user_id']:>3}    {str(row['event_ts'].date()):>12}  "
          f"{pit:>12}  {naive:>14}  {'✓' if match else '✗ LEAK'}")

### 2.3 Feature registry simulation

In [ ]:
import hashlib
import datetime
from typing import Callable, Any

class FeatureDefinition:
    def __init__(self, name: str, compute_fn: Callable,
                 max_staleness_seconds: int, version: int = 1):
        self.name = name
        self.compute_fn = compute_fn
        self.max_staleness_seconds = max_staleness_seconds
        self.version = version
        # Signature is the hash of the function source — changing logic → new hash
        self.signature = hashlib.md5(
            compute_fn.__code__.co_code
        ).hexdigest()[:8]

    def compute(self, raw_features: dict) -> Any:
        return self.compute_fn(raw_features)

    def check_freshness(self, computed_at: datetime.datetime) -> bool:
        age = (datetime.datetime.now(datetime.timezone.utc) - computed_at).total_seconds()
        return age <= self.max_staleness_seconds

# Define features ONCE — shared by training pipeline and serving layer
FEATURE_REGISTRY = {
    'spend_capped': FeatureDefinition(
        name='spend_capped',
        compute_fn=lambda raw: min(raw.get('spend_sgd', 0.0), 38.19),
        max_staleness_seconds=3600,
    ),
    'engagement_rate': FeatureDefinition(
        name='engagement_rate',
        compute_fn=lambda raw: raw.get('watch_hours_week', 0) / max(raw.get('days_active', 1), 1),
        max_staleness_seconds=3600,
    ),
    'tenure_bucket': FeatureDefinition(
        name='tenure_bucket',
        compute_fn=lambda raw: min(int(raw.get('tenure_months', 0) / 6), 4),
        max_staleness_seconds=86400,
    ),
}

def compute_features(raw: dict, registry: dict) -> dict:
    return {name: defn.compute(raw) for name, defn in registry.items()}

# --- Same registry used in TRAINING ---
training_raw = {'spend_sgd': 85.0, 'watch_hours_week': 22, 'days_active': 20, 'tenure_months': 14}
training_features = compute_features(training_raw, FEATURE_REGISTRY)
print("Training features:")
for k, v in training_features.items():
    print(f"  {k:20s}: {v}")

# --- Same registry used in SERVING ---
serving_raw = {'spend_sgd': 85.0, 'watch_hours_week': 22, 'days_active': 20, 'tenure_months': 14}
serving_features = compute_features(serving_raw, FEATURE_REGISTRY)
print("\nServing features (same registry call):")
for k, v in serving_features.items():
    print(f"  {k:20s}: {v}")

print(f"\nFeatures match: {training_features == serving_features}")
print("\nFeature signatures (code hash — changes if logic changes):")
for name, defn in FEATURE_REGISTRY.items():
    print(f"  {name:20s}: v{defn.version}  sig={defn.signature}  "
          f"max_staleness={defn.max_staleness_seconds}s")

## 3. CinemaStream in Practice

```python
# Ch077 serve.py — the training-serving skew liability
# SPEND_CAP = 38.1925   # 95th percentile from training — FROZEN
# SPEND_MEDIANS = {'Free': 0.005, 'Basic': 12.91, 'Premium': 19.945}
```

In [ ]:
import numpy as np
import pandas as pd

# Canonical constants from Ch077 training (frozen at promotion)
SPEND_CAP_TRAINING   = 38.1925
SPEND_MEDIAN_FREE    = 0.005
SPEND_MEDIAN_BASIC   = 12.91
SPEND_MEDIAN_PREMIUM = 19.945

# What the constants would be if recomputed from today's larger dataset
# (simulated: 2 extra months of data shifted the 95th percentile and medians)
SPEND_CAP_CURRENT    = 41.20   # new export has more high-spend Premium users
SPEND_MEDIAN_PREMIUM_CURRENT = 21.30  # Premium median drifted up with price increase

# Test with 5 representative serving requests
test_users = pd.DataFrame({
    'user_id':         [1001, 1002, 1003, 1004, 1005],
    'plan':            ['Premium', 'Free', 'Basic', 'Premium', 'Free'],
    'spend_raw':       [45.0,  0.0, 13.5, 22.0,  0.0],
    'watch_hrs_week':  [18.0,  4.5, 11.0, 25.0,  2.0],
    'days_active':     [22,    6,   15,   28,    3],
})

def engineer_features_frozen(row):
    """Current serve.py — uses frozen training constants."""
    spend_median = {'Free': SPEND_MEDIAN_FREE, 'Basic': SPEND_MEDIAN_BASIC,
                    'Premium': SPEND_MEDIAN_PREMIUM}[row['plan']]
    spend = row['spend_raw'] if row['spend_raw'] > 0 else spend_median
    return {
        'spend_capped': min(spend, SPEND_CAP_TRAINING),
        'engagement':   row['watch_hrs_week'] / max(row['days_active'], 1),
    }

def engineer_features_updated(row):
    """What the constants WOULD be if we updated them for the new dataset."""
    spend_median = {'Free': SPEND_MEDIAN_FREE, 'Basic': SPEND_MEDIAN_BASIC,
                    'Premium': SPEND_MEDIAN_PREMIUM_CURRENT}[row['plan']]
    spend = row['spend_raw'] if row['spend_raw'] > 0 else spend_median
    return {
        'spend_capped': min(spend, SPEND_CAP_CURRENT),
        'engagement':   row['watch_hrs_week'] / max(row['days_active'], 1),
    }

print("Feature drift check: frozen constants vs would-be-updated constants")
print(f"{'user':>6}  {'plan':>8}  "
      f"{'spend_frozen':>13}  {'spend_current':>14}  {'delta':>8}")
for _, row in test_users.iterrows():
    frozen  = engineer_features_frozen(row)
    current = engineer_features_updated(row)
    delta   = current['spend_capped'] - frozen['spend_capped']
    print(f"  {row['user_id']:>4}  {row['plan']:>8}  "
          f"  {frozen['spend_capped']:>10.2f}    {current['spend_capped']:>11.2f}  "
          f"  {delta:>+6.2f}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

---

# Chapter 77b: Time-Series Forecasting for Business Metrics

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas scikit-learn

### 2.1 Decomposing a Trend by Hand

In [ ]:
import numpy as np
import pandas as pd

# Synthetic monthly series: 18 months with an upward trend + seasonality + noise
rng = np.random.default_rng(42)
months = pd.date_range("2024-01", periods=18, freq="MS")
trend_component = np.linspace(40_000, 58_000, 18)            # upward trend
seasonality = 3_000 * np.sin(np.linspace(0, 2 * np.pi, 18)) # annual cycle
noise = rng.normal(0, 800, 18)
values = trend_component + seasonality + noise

series = pd.Series(values, index=months, name="metric")

# Rolling mean to isolate trend (window=3 smooths monthly noise)
rolling_trend = series.rolling(window=3, center=True).mean()

# Detrended residual
detrended = series - rolling_trend.fillna(series.mean())

print("Original (first 6):")
print(series.head(6).round(0))
print("\nSmoothed trend (first 6):")
print(rolling_trend.head(6).round(0))
print("\nDetrended residual std:", round(detrended.std(), 0))

### 2.2 The Temporal Leakage Trap

In [ ]:
# Simulate random split vs chronological split error
rng = np.random.default_rng(0)
n = 24
t = np.arange(n)
y = 0.5 * t + rng.normal(0, 1.5, n)  # upward trend + noise

# --- WRONG: random split leaks future into training ---
idx = rng.permutation(n)
train_idx_random = np.sort(idx[:18])
test_idx_random  = np.sort(idx[18:])

# --- CORRECT: chronological split ---
train_idx_chron = np.arange(18)
test_idx_chron  = np.arange(18, 24)

def rolling_mean_forecast(train_y, n_ahead, window=3):
    """Predict n_ahead steps using the rolling mean of the last `window` values."""
    preds = []
    history = list(train_y)
    for _ in range(n_ahead):
        pred = np.mean(history[-window:])
        preds.append(pred)
        history.append(pred)
    return np.array(preds)

# Random-split MAE (cheating: model trained on future months)
y_train_r = y[train_idx_random]
y_test_r  = y[test_idx_random]
preds_r   = np.mean(y_train_r[-3:]) * np.ones(len(test_idx_random))
mae_random = np.mean(np.abs(preds_r - y_test_r))

# Chronological MAE (honest)
y_train_c = y[train_idx_chron]
y_test_c  = y[test_idx_chron]
preds_c   = rolling_mean_forecast(y_train_c, len(test_idx_chron), window=3)
mae_chron  = np.mean(np.abs(preds_c - y_test_c))

print(f"Random-split MAE (leaks future):  {mae_random:.2f}")
print(f"Chronological MAE (honest):       {mae_chron:.2f}")

### 2.3 Baselines: Last-Value and Rolling Mean

In [ ]:
rng = np.random.default_rng(7)
n_train, n_test = 15, 3
t = np.arange(n_train + n_test)
y = 2.0 * t + rng.normal(0, 3.0, len(t))

y_train = y[:n_train]
y_test  = y[n_train:]

# Baseline 1: last-value (naïve)
pred_last = np.full(n_test, y_train[-1])

# Baseline 2: rolling mean (window=3)
pred_roll = rolling_mean_forecast(y_train, n_test, window=3)

# Baseline 3: linear trend extrapolation
x_train = np.arange(n_train)
m, b    = np.polyfit(x_train, y_train, 1)
x_test  = np.arange(n_train, n_train + n_test)
pred_trend = m * x_test + b

mae_last  = np.mean(np.abs(pred_last  - y_test))
mae_roll  = np.mean(np.abs(pred_roll  - y_test))
mae_trend = np.mean(np.abs(pred_trend - y_test))

print(f"Last-value baseline MAE:      {mae_last:.1f}")
print(f"Rolling-mean baseline MAE:    {mae_roll:.1f}")
print(f"Linear trend extrapolation:   {mae_trend:.1f}")
print(f"\nTrue next-3 values:  {y_test.round(1)}")
print(f"Last-value forecast: {pred_last.round(1)}")
print(f"Rolling-mean:        {pred_roll.round(1)}")
print(f"Trend forecast:      {pred_trend.round(1)}")

### 2.4 Holt's Exponential Smoothing (Manual)

In [ ]:
def holts_exponential_smoothing(y, alpha, beta, n_ahead=1):
    """
    Holt's double exponential smoothing.
    alpha: smoothing for level  (0=ignore new data, 1=ignore history)
    beta:  smoothing for trend  (0=ignore new trend, 1=ignore trend history)
    """
    level = y[0]
    trend = y[1] - y[0]  # initial trend estimate

    levels, trends = [level], [trend]

    for obs in y[1:]:
        prev_level = level
        level = alpha * obs + (1 - alpha) * (prev_level + trend)
        trend = beta * (level - prev_level) + (1 - beta) * trend
        levels.append(level)
        trends.append(trend)

    forecasts = [level + (h + 1) * trend for h in range(n_ahead)]
    return np.array(forecasts), np.array(levels), np.array(trends)

# Toy: monthly active users with upward trend
rng = np.random.default_rng(12)
y_mau = 50_000 + 2_000 * np.arange(12) + rng.normal(0, 1_500, 12)

# Compare alpha/beta combinations
for alpha, beta in [(0.3, 0.1), (0.6, 0.3), (0.9, 0.5)]:
    preds, _, _ = holts_exponential_smoothing(y_mau[:9], alpha, beta, n_ahead=3)
    mae = np.mean(np.abs(preds - y_mau[9:]))
    print(f"alpha={alpha}, beta={beta} → 3-step MAE: {mae:,.0f}  "
          f"| forecasts: {preds.round(0)}")

print(f"\nActual months 10-12: {y_mau[9:].round(0)}")

### 2.5 TimeSeriesSplit: Backtesting Properly

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

rng = np.random.default_rng(99)
n = 24
y_full = 100 + 3 * np.arange(n) + rng.normal(0, 5, n)

tscv = TimeSeriesSplit(n_splits=4)
fold_maes = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(y_full)):
    y_tr = y_full[train_idx]
    y_te = y_full[test_idx]

    preds, _, _ = holts_exponential_smoothing(y_tr, alpha=0.5, beta=0.2,
                                               n_ahead=len(y_te))
    mae = np.mean(np.abs(preds - y_te))
    fold_maes.append(mae)
    print(f"Fold {fold+1}: train months 1–{len(y_tr):2d} | "
          f"test months {len(y_tr)+1:2d}–{len(y_tr)+len(y_te):2d} | MAE {mae:.1f}")

print(f"\nMean MAE across folds: {np.mean(fold_maes):.1f}  "
      f"(±{np.std(fold_maes):.1f})")

### 2.6 Communicating Forecast Uncertainty

In [ ]:
# Build a forecast interval from backtesting residuals
rng = np.random.default_rng(77)
n_train = 18
y_long = 200 + 4 * np.arange(24) + rng.normal(0, 6, 24)
y_tr = y_long[:n_train]
y_te = y_long[n_train:]

# 1-step-ahead residuals on training data (walk-forward)
residuals = []
for i in range(3, n_train):
    pred_1, _, _ = holts_exponential_smoothing(y_tr[:i], alpha=0.5, beta=0.2,
                                                n_ahead=1)
    residuals.append(pred_1[0] - y_tr[i])

residuals = np.array(residuals)
std_error = residuals.std()

# Forecast next 6 months with ±2σ interval
preds_6, _, _ = holts_exponential_smoothing(y_tr, alpha=0.5, beta=0.2, n_ahead=6)
lower = preds_6 - 2 * std_error
upper = preds_6 + 2 * std_error

print("6-month forecast with 95% prediction interval:")
print(f"{'Month':<6}  {'Forecast':>10}  {'Lower (−2σ)':>12}  {'Upper (+2σ)':>12}  {'Actual':>10}")
for i, (p, lo, hi, act) in enumerate(zip(preds_6, lower, upper, y_te)):
    inside = "✓" if lo <= act <= hi else "✗"
    print(f"  +{i+1:<4} {p:>10.0f}  {lo:>12.0f}  {hi:>12.0f}  {act:>10.0f}  {inside}")

covered = np.sum((y_te >= lower) & (y_te <= upper))
print(f"\n{covered}/6 actuals inside 95% interval  "
      f"(expect ~5-6 for a well-calibrated interval)")

## 3. CinemaStream in Practice

In [ ]:
import numpy as np
import pandas as pd

# CinemaStream monthly business metrics — 2024-01 through 2025-06 (18 months)
# Invented values (canonical — logged in session_log.md)
months = pd.date_range("2024-01", periods=18, freq="MS")

mrr_sgd = np.array([
    58_200, 60_100, 62_800, 61_400, 63_900, 66_200,   # Q1–Q2 2024
    68_500, 67_200, 70_100, 72_400, 73_800, 71_600,   # Q3–Q4 2024
    74_900, 76_300, 78_100, 79_600, 81_200, 83_500,   # Q1–Q2 2025
])

watch_minutes_M = np.array([
     82.1,  84.5,  88.3,  87.2,  91.6,  95.1,
     98.7,  97.4, 103.2, 106.1, 108.4, 104.9,
    109.3, 111.8, 114.2, 116.5, 118.9, 122.1,
])

support_tickets = np.array([
    410, 398, 421, 445, 433, 460,
    489, 501, 477, 512, 525, 540,
    558, 547, 571, 589, 602, 618,
])

# Use the last 15 months for training, hold out final 3 for validation
n_train = 15
train_months = months[:n_train]
val_months   = months[n_train:]

def forecast_with_interval(series, n_train, n_ahead, alpha=0.5, beta=0.2):
    """Holt's forecast + ±2σ interval from walk-forward residuals."""
    y_tr = series[:n_train]
    y_va = series[n_train:]

    # Walk-forward residuals on training data
    residuals = []
    for i in range(4, n_train):
        p, _, _ = holts_exponential_smoothing(y_tr[:i], alpha, beta, n_ahead=1)
        residuals.append(p[0] - y_tr[i])

    std_err = np.std(residuals)
    preds, _, _ = holts_exponential_smoothing(y_tr, alpha, beta, n_ahead=n_ahead)
    lower = preds - 2 * std_err
    upper = preds + 2 * std_err

    val_mae = np.mean(np.abs(preds[:len(y_va)] - y_va)) if len(y_va) else np.nan
    return preds, lower, upper, val_mae, std_err

print("=== CinemaStream 6-Month Forecast (Jul–Dec 2025) ===\n")

for name, series, unit in [
    ("MRR (SGD)", mrr_sgd, "S$"),
    ("Watch Minutes (M)", watch_minutes_M, ""),
    ("Support Tickets", support_tickets, ""),
]:
    preds, lower, upper, val_mae, std_err = forecast_with_interval(
        series, n_train=n_train, n_ahead=6, alpha=0.5, beta=0.2
    )
    val_actual = series[n_train:]
    covered = np.sum((val_actual >= lower[:3]) & (val_actual <= upper[:3]))
    print(f"{name}")
    print(f"  Validation MAE (last 3 months): {unit}{val_mae:,.1f}")
    print(f"  Residual σ: {unit}{std_err:,.1f}  |  Validation coverage: {covered}/3")
    print(f"  6-Month Forecasts:")
    future_months = pd.date_range("2025-07", periods=6, freq="MS")
    for m, p, lo, hi in zip(future_months, preds, lower, upper):
        print(f"    {m.strftime('%b %Y')}: {unit}{p:,.0f}  "
              f"[{unit}{lo:,.0f} – {unit}{hi:,.0f}]")
    print()

In [ ]:
# The one insight that changes a meeting: YoY growth rate for tickets
tickets_jun_2024 = support_tickets[5]   # June 2024
tickets_jun_2025 = support_tickets[17]  # June 2025 (most recent)
yoy_pct = (tickets_jun_2025 - tickets_jun_2024) / tickets_jun_2024 * 100

tickets_dec_2025_forecast = 637  # midpoint from above
tickets_dec_2024 = support_tickets[11]  # December 2024
forecast_yoy_pct = (tickets_dec_2025_forecast - tickets_dec_2024) / tickets_dec_2024 * 100

print(f"Support ticket YoY (Jun 2024 → Jun 2025): +{yoy_pct:.0f}%")
print(f"Forecast YoY (Dec 2024 → Dec 2025):       +{forecast_yoy_pct:.0f}%")
print()
print("MRR forecast summary for board slide:")
print("  Dec 2025 central estimate:  S$85,851")
print("  95% prediction interval:    S$83,028 – S$88,673")
print("  What the interval excludes: one-off price changes, competitor moves,")
print("                              macro shocks, major content licensing events")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import numpy as np

def holts_exponential_smoothing(y, alpha, beta, n_ahead=1):
    """
    Holt's double exponential smoothing.
    alpha: smoothing for level  (0=ignore new data, 1=ignore history)
    beta:  smoothing for trend  (0=ignore new trend, 1=ignore trend history)
    """
    level = y[0]
    trend = y[1] - y[0]  # initial trend estimate

    levels, trends = [level], [trend]

    for obs in y[1:]:
        prev_level = level
        level = alpha * obs + (1 - alpha) * (prev_level + trend)
        trend = beta * (level - prev_level) + (1 - beta) * trend
        levels.append(level)
        trends.append(trend)

    forecasts = [level + (h + 1) * trend for h in range(n_ahead)]
    return np.array(forecasts), np.array(levels), np.array(trends)

rng = np.random.default_rng(3)
y = 5 * np.arange(20) + rng.normal(0, 8, 20)
y_tr, y_te = y[:16], y[16:]

pred_last = np.full(4, y_tr[-1])
pred_roll = np.full(4, np.mean(y_tr[-3:]))
pred_holts_1, _, _ = holts_exponential_smoothing(y_tr, 0.4, 0.2, n_ahead=4)
pred_holts_2, _, _ = holts_exponential_smoothing(y_tr, 0.8, 0.5, n_ahead=4)

for name, pred in [
    ("Last-value", pred_last),
    ("Rolling-mean", pred_roll),
    ("Holt's (0.4, 0.2)", pred_holts_1),
    ("Holt's (0.8, 0.5)", pred_holts_2),
]:
    mae = np.mean(np.abs(pred - y_te))
    print(f"{name:<22} MAE = {mae:.1f}")

print(f"\nActual: {y_te.round(1)}")

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

def holts_exponential_smoothing(y, alpha, beta, n_ahead=1):
    """
    Holt's double exponential smoothing.
    alpha: smoothing for level  (0=ignore new data, 1=ignore history)
    beta:  smoothing for trend  (0=ignore new trend, 1=ignore trend history)
    """
    level = y[0]
    trend = y[1] - y[0]  # initial trend estimate

    levels, trends = [level], [trend]

    for obs in y[1:]:
        prev_level = level
        level = alpha * obs + (1 - alpha) * (prev_level + trend)
        trend = beta * (level - prev_level) + (1 - beta) * trend
        levels.append(level)
        trends.append(trend)

    forecasts = [level + (h + 1) * trend for h in range(n_ahead)]
    return np.array(forecasts), np.array(levels), np.array(trends)

mrr_sgd = np.array([
    58_200, 60_100, 62_800, 61_400, 63_900, 66_200,
    68_500, 67_200, 70_100, 72_400, 73_800, 71_600,
    74_900, 76_300, 78_100, 79_600, 81_200, 83_500,
])

tscv = TimeSeriesSplit(n_splits=3)
ts_maes = []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(mrr_sgd)):
    preds, _, _ = holts_exponential_smoothing(
        mrr_sgd[tr_idx], alpha=0.5, beta=0.2, n_ahead=len(te_idx)
    )
    mae = np.mean(np.abs(preds - mrr_sgd[te_idx]))
    ts_maes.append(mae)
    print(f"Fold {fold+1}: train n={len(tr_idx)}, test n={len(te_idx)}, MAE=S${mae:,.0f}")

# Random split (leaking)
rng = np.random.default_rng(0)
idx = rng.permutation(18)
tr_r, te_r = mrr_sgd[idx[:14]], mrr_sgd[idx[14:]]
preds_r = np.full(4, np.mean(tr_r[-3:]))  # rolling mean on random subset
mae_random = np.mean(np.abs(preds_r - te_r))

print(f"\nTimeSeriesSplit mean MAE: S${np.mean(ts_maes):,.0f}")
print(f"Random-split MAE:         S${mae_random:,.0f}")
print(f"Leakage makes it look {np.mean(ts_maes) / mae_random:.1f}x better")